# FreightQuote AI — Final Colab Notebook
Run Steps 1–6 in order using your existing Colab Secrets. Includes the complete application, authentication module, styling, image and tests.

Step 6 reuses an existing local tunnel. If another runtime holds your ngrok address, it offers a Colab preview and explains how to release that endpoint.

This is a Colab demonstration deployment. For permanent hosting, use a persistent database volume and a continuously running host; Colab storage and public tunnels are temporary.


## 1. Install dependencies

In [ ]:
%pip -q install "streamlit==1.66.1" "bcrypt==5.0.0" "PyJWT==2.15.1" "pandas==2.2.3" "pyngrok>=7,<8"


## 2. Configure Secrets

Enable notebook access for `JWT_SECRET`, `NGROK_AUTHTOKEN`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD` and `ADMIN_PASSWORD`. `ADMIN_USERNAME` is optional and defaults to `admin`. Your existing values can be reused.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

required = ("JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_PASSWORD")
missing = []
for name in required:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value or not str(value).strip():
        missing.append(name)
    else:
        os.environ[name] = str(value).strip()
if missing:
    raise ValueError("Add these Colab Secrets and enable access: " + ", ".join(missing))
if len(os.environ["JWT_SECRET"]) < 32:
    raise ValueError("JWT_SECRET must be at least 32 characters.")
if not 12 <= len(os.environ["ADMIN_PASSWORD"].encode("utf-8")) <= 72:
    raise ValueError("ADMIN_PASSWORD must be 12–72 UTF-8 bytes.")
try:
    admin_name = userdata.get("ADMIN_USERNAME")
except Exception:
    admin_name = None
os.environ["ADMIN_USERNAME"] = str(admin_name).strip() if admin_name else "admin"
os.environ["EMAIL_PASSWORD"] = os.environ["EMAIL_PASSWORD"].replace(" ", "")
os.environ["REQUIRE_EMAIL_VERIFICATION"] = "1"
os.environ["DATABASE_PATH"] = "/content/portwise_auth.db"
os.environ["SHOW_BUILD_ID"] = "0"
project_dir = Path("/content/FreightQuote_AI_Lite")
project_dir.mkdir(parents=True, exist_ok=True)
os.chdir(project_dir)
Path("tests").mkdir(exist_ok=True)
Path("screenshots").mkdir(exist_ok=True)
Path(".streamlit").mkdir(exist_ok=True)
print("Configuration ready.")


## 3. Create all application files

Run the single cell below. Wait for **Setup complete** before continuing. This writes the application, styles, tests and photo together.

In [ ]:
#@title 3. Create ALL application files and artwork (run this one cell)
from pathlib import Path
import base64

PROJECT_DIR = Path("/content/FreightQuote_AI_Lite")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
APP_FILES = {'app.py': '"""FreightQuote AI | Maritime freight workspace · Milestone 1 authentication."""\n\nimport time\nimport html\nimport os\nimport smtplib\nfrom email.message import EmailMessage\nfrom pathlib import Path\n\nimport pandas as pd\nimport streamlit as st\nfrom auth_core import (\n    QUESTIONS,\n    ROLES,\n    can,\n    admin_credential_tag,\n    Store,\n    decode_token,\n    issue_token,\n    password_errors,\n    valid_email,\n    answer_error,\n    LOCKED_MESSAGE,\n)\n\nst.set_page_config(\n    page_title="FreightQuote AI | Maritime Intelligence",\n    page_icon="⚓",\n    layout="wide",\n    initial_sidebar_state="auto",\n)\n\nJWT_SECRET = os.getenv("JWT_SECRET", "")\nEMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "")\nEMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")\nADMIN_USERNAME = os.getenv("ADMIN_USERNAME", "admin")\nADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "")\nREQUIRE_EMAIL_VERIFICATION = os.getenv(\n    "REQUIRE_EMAIL_VERIFICATION", "1"\n).lower() not in ("0", "false", "no")\nstore = Store(os.getenv("DATABASE_PATH", "auth.db"))\nBASE_DIR = Path(__file__).resolve().parent\n\n\nROUTES = {\n    "login",\n    "locked",\n    "signup",\n    "forgot",\n    "dashboard",\n    "admin-dashboard",\n    "verify-email",\n}\nrequested = str(st.query_params.get("view", "login"))\nif requested not in ROUTES:\n    requested = "login"\nfor key, initial in {\n    "view": requested,\n    "theme": "Light",\n    "motion_enabled": True,\n    "jwt": None,\n    "reset_verified": False,\n    "reset_identity": None,\n    "reset_method": None,\n    "question_user": None,\n    "flash": None,\n    "reset_grant": None,\n    "verify_email_address": None,\n}.items():\n    if key not in st.session_state:\n        st.session_state[key] = initial\nif st.session_state.view != requested:\n    st.session_state.view = requested\n    st.session_state.reset_verified = False\n    st.session_state.reset_identity = None\n    st.session_state.question_user = None\n    st.session_state.reset_grant = None\n\nDARK = st.session_state.theme == "Dark"\nPALETTE = (\n    {\n        "bg": "#101522",\n        "card": "#182032",\n        "ink": "#eef3ff",\n        "sub": "#aebbd1",\n        "line": "#303d54",\n        "accent": "#79d5cd",\n        "soft": "#242e46",\n    }\n    if DARK\n    else {\n        "bg": "#f5f7fb",\n        "card": "#ffffff",\n        "ink": "#202b43",\n        "sub": "#617087",\n        "line": "#e0e6f0",\n        "accent": "#167b83",\n        "soft": "#eef0f9",\n    }\n)\ncss_variables = (\n    ":root{"\n    + ";".join("--pw-" + key + ":" + value for key, value in PALETTE.items())\n    + "}"\n)\nst.markdown(\n    "<style>" + css_variables + (BASE_DIR / "motion.css").read_text() + "</style>",\n    unsafe_allow_html=True,\n)\nif not st.session_state.motion_enabled:\n    st.markdown(\n        "<style>*,*::before,*::after{animation:none!important;transition:none!important;scroll-behavior:auto!important}</style>",\n        unsafe_allow_html=True,\n    )\n\n\ndef navigate(view):\n    if view != st.session_state.view:\n        for key in ("reset_identity", "question_user", "reset_grant"):\n            st.session_state[key] = None\n        st.session_state.reset_verified = False\n    st.session_state.view = view\n    st.query_params["view"] = view\n    st.rerun()\n\n\ndef flash():\n    if st.session_state.flash:\n        st.success(st.session_state.flash)\n        st.session_state.flash = None\n\n\ndef current_user():\n    return store.session(\n        st.session_state.jwt, JWT_SECRET, ADMIN_USERNAME, ADMIN_PASSWORD\n    )\n\n\ndef log_out():\n    account = current_user()\n    if account:\n        store.audit(account["username"], "logout")\n    for key in list(st.session_state):\n        del st.session_state[key]\n    st.query_params["view"] = "login"\n    st.rerun()\n\n\ndef password_ok(password, confirmation):\n    missing = password_errors(password)\n    if missing:\n        st.error("Password must contain " + ", ".join(missing) + ".")\n        return False\n    if password != confirmation:\n        st.error("Passwords do not match.")\n        return False\n    return True\n\n\ndef send_code(recipient, code, purpose="reset"):\n    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:\n        return False, "Email delivery is unavailable. Contact the administrator."\n    mail = EmailMessage()\n    mail["From"] = EMAIL_ADDRESS\n    mail["To"] = recipient\n    title = (\n        "Verify your FreightQuote AI email"\n        if purpose == "signup"\n        else "FreightQuote AI password recovery"\n    )\n    mail["Subject"] = title\n    mail.set_content(\n        f"Your FreightQuote AI six-digit code is {code}. It expires in five minutes. If you didn\'t request it, ignore this email."\n    )\n    try:\n        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as smtp:\n            smtp.login(EMAIL_ADDRESS.strip(), EMAIL_PASSWORD.replace(" ", ""))\n            refused = smtp.send_message(mail)\n            if refused:\n                return (\n                    False,\n                    "The email service refused the recipient. Check the address and try again.",\n                )\n        return True, None\n    except smtplib.SMTPAuthenticationError:\n        return (\n            False,\n            "Gmail rejected the sender login. Verify the Gmail App Password in Colab Secrets.",\n        )\n    except (smtplib.SMTPException, OSError):\n        return (\n            False,\n            "Email sending failed. Check the sender settings and SMTP connectivity in Colab.",\n        )\n\n\ndef topbar():\n    with st.container(key="topbar"):\n        left, theme_control = st.columns(\n            [8, 1.2], vertical_alignment="center"\n        )\n        with left:\n            st.markdown(\n                \'<div class="top-brand"><span class="brand-symbol">FQ</span> FreightQuote AI <small>FREIGHT INTELLIGENCE</small></div>\',\n                unsafe_allow_html=True,\n            )\n        with theme_control:\n            if st.button("☀ Light" if DARK else "☾ Dark", width="stretch"):\n                st.session_state.theme = "Light" if DARK else "Dark"\n                st.rerun()\n\n\ndef left_panel():\n    st.markdown(\n        """<section class="auth-hero freight-scene">\n<div class="freight-photo"></div><div class="freight-shade"></div>\n<div class="freight-copy"><div class="hero-eyebrow">FREIGHT INTELLIGENCE, CONNECTED</div>\n<h1>A world in motion.<br><em>A smarter way forward.</em></h1>\n<p class="hero-copy">Your freight team. One connected workspace.</p></div>\n</section>""",\n        unsafe_allow_html=True,\n    )\n\n\ndef login():\n    st.markdown(\'<div class="fq-intro"><p class="fq-kicker">Sign in</p><h3>Welcome back</h3></div>\', unsafe_allow_html=True)\n    st.caption("Sign in to your FreightQuote AI workspace.")\n    flash()\n    with st.form("login_form"):\n        identity = st.text_input(\n            "Username or email",\n            placeholder="Enter your username or email",\n            max_chars=254,\n        )\n        password = st.text_input(\n            "Password", type="password", placeholder="Enter your password", max_chars=72\n        )\n        submit = st.form_submit_button("Sign in →", type="primary", width="stretch")\n    if submit:\n        if not identity.strip() or not password:\n            st.error("Both fields are required.")\n        else:\n            account, error = store.sign_in(\n                identity, password, ADMIN_USERNAME, ADMIN_PASSWORD\n            )\n            if error:\n                st.error(error)\n            else:\n                st.session_state.jwt = issue_token(\n                    account["email"] or account["username"],\n                    JWT_SECRET,\n                    role=account["role"],\n                    version=account["session_version"],\n                    kind=account["principal_type"],\n                    credential_tag=(\n                        admin_credential_tag(ADMIN_PASSWORD, JWT_SECRET)\n                        if account["principal_type"] == "configured_admin"\n                        else ""\n                    ),\n                )\n                navigate("dashboard")\n    a, b = st.columns(2)\n    with a:\n        if st.button("Create account", width="stretch"):\n            navigate("signup")\n    with b:\n        if st.button("Forgot password?", width="stretch"):\n            navigate("forgot")\n    with st.expander("Verify your email"):\n        if st.button("Verify a registered email address", width="stretch"):\n            navigate("verify-email")\n\n\ndef signup():\n    st.markdown(\'<div class="fq-intro"><p class="fq-kicker">Create account</p><h3>Join the workspace</h3></div>\', unsafe_allow_html=True)\n    st.caption(\n        "Create your customer account. Your administrator can assign team access."\n    )\n    with st.form("signup_form"):\n        username = st.text_input("Username", max_chars=50)\n        email = st.text_input("Email address", max_chars=254)\n        a, b = st.columns(2)\n        with a:\n            password = st.text_input("Password", type="password", max_chars=72)\n        with b:\n            confirm = st.text_input("Confirm password", type="password", max_chars=72)\n        question = st.selectbox(\n            "Security question", QUESTIONS, index=None, placeholder="Choose a question"\n        )\n        answer = st.text_input("Security answer", type="password", max_chars=72)\n        st.caption("8+ characters: uppercase, lowercase, number and symbol. No spaces.")\n        submit = st.form_submit_button(\n            "Create account →", type="primary", width="stretch"\n        )\n    if submit:\n        if not all(\n            (\n                username.strip(),\n                email.strip(),\n                password,\n                confirm,\n                question,\n                answer.strip(),\n            )\n        ):\n            st.error("Complete all required fields.")\n        elif answer_error(answer):\n            st.error(answer_error(answer))\n        elif not valid_email(email):\n            st.error("Enter a valid email, such as ab@cd.ef.")\n        elif password_ok(password, confirm):\n            if REQUIRE_EMAIL_VERIFICATION and not (EMAIL_ADDRESS and EMAIL_PASSWORD):\n                st.error("Email delivery is unavailable. Contact the administrator.")\n                return\n            ok, message = store.signup(\n                username,\n                email,\n                password,\n                question,\n                answer,\n                require_verification=REQUIRE_EMAIL_VERIFICATION,\n                reserved_identity=ADMIN_USERNAME,\n            )\n            if ok:\n                if REQUIRE_EMAIL_VERIFICATION:\n                    st.session_state.verify_email_address = email.strip().lower()\n                    sent = deliver_code(email.strip().lower(), "verify")\n                    st.session_state.flash = (\n                        "Account created. Check your email for the code."\n                        if sent\n                        else "Account created. Email delivery failed; request a new code after the cooldown."\n                    )\n                    navigate("verify-email")\n                else:\n                    st.session_state.flash = message\n                    navigate("login")\n            else:\n                st.error(message)\n    if st.button("← Back to sign in"):\n        navigate("login")\n\n\ndef close_recovery():\n    """End the recovery flow and show one account-lock notice."""\n    for key in (\n        "reset_identity",\n        "reset_grant",\n        "question_user",\n        "verify_email_address",\n    ):\n        st.session_state[key] = None\n    st.session_state.reset_verified = False\n    st.session_state.view = "locked"\n    st.query_params["view"] = "locked"\n    st.rerun(scope="app")\n\n\ndef locked_page():\n    st.markdown(\'<div class="fq-intro"><p class="fq-kicker">Account access</p><h3>Account locked</h3></div>\', unsafe_allow_html=True)\n    st.error(LOCKED_MESSAGE)\n    if st.button("Back to sign in", key="locked_back"):\n        navigate("login")\n\n\ndef deliver_code(address, purpose="reset"):\n    if store.is_locked(address):\n        close_recovery()\n    begin = store.begin_otp if purpose == "reset" else store.begin_email_verification\n    code, message = begin(address, JWT_SECRET)\n    if not code:\n        if store.is_locked(address):\n            close_recovery()\n        st.info(message)\n        return False\n    sent, error = send_code(address, code, "reset" if purpose == "reset" else "signup")\n    if not sent:\n        cancel = (\n            store.cancel_otp if purpose == "reset" else store.cancel_email_verification\n        )\n        cancel(address)\n        st.error(error)\n        return False\n    st.success("Code sent. Check your inbox and spam folder.")\n    return True\n\n\n@st.fragment(run_every="1s")\ndef code_panel(address, purpose="reset"):\n    status = store.code_status(address, purpose)\n    if status["locked"]:\n        close_recovery()\n        return\n    expiry = status["expires_in"]\n    cooldown = status["resend_in"]\n    minutes, seconds = divmod(expiry, 60)\n    st.caption(\n        f"Code expires in {minutes:02d}:{seconds:02d}"\n        if expiry\n        else "No active code. Request a new one when available."\n    )\n    if cooldown:\n        minutes, seconds = divmod(cooldown, 60)\n        st.caption(f"Resend available in {minutes:02d}:{seconds:02d}")\n    st.caption(\n        f"Code request {status[\'requests\']} of 5 · Incorrect entries {status[\'failures\']} of 5"\n    )\n    if st.button(\n        "Resend code",\n        key="resend_" + purpose,\n        disabled=cooldown > 0 or status["requests"] >= 5,\n        width="stretch",\n    ):\n        deliver_code(address, purpose)\n        st.rerun()\n    if status["requests"] >= 5:\n        st.caption("This is your last code. If it expires, contact the administrator.")\n\n\ndef verify_email_page():\n    st.markdown(\'<div class="fq-intro"><p class="fq-kicker">Email verification</p><h3>Check your inbox</h3></div>\', unsafe_allow_html=True)\n    st.caption("Enter the code sent to your registered email.")\n    flash()\n    with st.form("email_verification_request"):\n        address = st.text_input(\n            "Registered email address",\n            value=st.session_state.verify_email_address or "",\n            max_chars=254,\n        )\n        request = st.form_submit_button("Send code", type="primary", width="stretch")\n    if request:\n        if not valid_email(address):\n            st.error("Enter a valid email address.")\n        else:\n            address = address.strip().lower()\n            if deliver_code(address, "verify"):\n                st.session_state.verify_email_address = address\n                st.rerun()\n    address = st.session_state.verify_email_address\n    if address:\n        status = store.code_status(address, "verify")\n        if status["requests"]:\n            code_panel(address, "verify")\n            with st.form("verify_signup_code"):\n                code = st.text_input("Six-digit code", max_chars=6)\n                confirm = st.form_submit_button(\n                    "Verify email", type="primary", width="stretch"\n                )\n            if confirm:\n                if not code.strip():\n                    st.error("Enter the six-digit code.")\n                elif store.verify_email(address, code.strip(), JWT_SECRET):\n                    st.session_state.verify_email_address = None\n                    st.session_state.flash = "Email verified. Sign in to continue."\n                    navigate("login")\n                elif store.is_locked(address):\n                    close_recovery()\n                else:\n                    st.error("Invalid or expired code.")\n    if st.button("Back to sign in", key="verify_back"):\n        navigate("login")\n\n\ndef forgot():\n    st.markdown(\'<div class="fq-intro"><p class="fq-kicker">Recovery</p><h3>Reset your password</h3></div>\', unsafe_allow_html=True)\n    method = st.radio(\n        "Recovery method", ["Email OTP", "Security question"], horizontal=True\n    )\n    kind = "otp" if method == "Email OTP" else "question"\n    if st.session_state.reset_method != kind:\n        st.session_state.reset_method = kind\n        st.session_state.reset_verified = False\n        st.session_state.reset_identity = None\n        st.session_state.question_user = None\n        st.session_state.reset_grant = None\n    address = st.session_state.reset_identity\n    if address and store.is_locked(address):\n        st.session_state.reset_grant = None\n        st.session_state.reset_verified = False\n        close_recovery()\n    elif st.session_state.reset_verified:\n        st.caption("Choose a new password. Verification is valid for five minutes.")\n        with st.form("new_password_form"):\n            pw = st.text_input("New password", type="password", max_chars=72)\n            confirm = st.text_input(\n                "Confirm new password", type="password", max_chars=72\n            )\n            submit = st.form_submit_button(\n                "Reset password", type="primary", width="stretch"\n            )\n        if submit and password_ok(pw, confirm):\n            ok, message = store.reset_password(\n                address, pw, st.session_state.reset_grant\n            )\n            if ok:\n                st.session_state.jwt = None\n                st.session_state.flash = message\n                navigate("login")\n            else:\n                st.error(message)\n                if "Verification expired" in message:\n                    st.session_state.reset_verified = False\n                    st.session_state.reset_grant = None\n        if st.button("Verify again"):\n            st.session_state.reset_verified = False\n            st.session_state.reset_grant = None\n            st.rerun()\n    elif kind == "otp":\n        st.caption(\n            "Codes expire after five minutes. Five incorrect entries lock your account."\n        )\n        st.caption(\n            "Resend delay: 1 minute for requests 1–3, then 5 minutes. Maximum 5 requests."\n        )\n        with st.form("send_otp_form"):\n            address = st.text_input("Registered email", max_chars=254)\n            send = st.form_submit_button("Send code", type="primary", width="stretch")\n        if send:\n            if not valid_email(address):\n                st.error("Enter a valid email address.")\n            elif not EMAIL_ADDRESS or not EMAIL_PASSWORD:\n                st.error("Email delivery is unavailable. Contact the administrator.")\n            else:\n                address = address.strip().lower()\n                sent = deliver_code(address)\n                # Restore the countdown from SQLite after a page refresh.\n                if sent:\n                    st.session_state.reset_identity = address\n                    st.rerun()\n                elif store.code_status(address)["requests"]:\n                    st.session_state.reset_identity = address\n        address = st.session_state.reset_identity\n        if address:\n            code_panel(address)\n            with st.form("verify_otp_form"):\n                code = st.text_input("Six-digit OTP", max_chars=6)\n                verify = st.form_submit_button(\n                    "Verify code", type="primary", width="stretch"\n                )\n            if verify:\n                if not code.strip():\n                    st.error("Enter the six-digit code.")\n                else:\n                    grant = store.verify_otp(address, code.strip(), JWT_SECRET)\n                    if grant:\n                        st.session_state.reset_grant = grant\n                        st.session_state.reset_verified = True\n                        st.rerun()\n                    elif store.is_locked(address):\n                        close_recovery()\n                    else:\n                        st.error("Invalid or expired code.")\n    else:\n        with st.form("find_question_form"):\n            username = st.text_input("Registered username", max_chars=50)\n            find = st.form_submit_button(\n                "Find security question", type="primary", width="stretch"\n            )\n        if find:\n            if not username.strip():\n                st.error("Enter your registered username.")\n            elif store.is_locked(username):\n                close_recovery()\n            elif store.security_question(username):\n                st.session_state.question_user = username.strip()\n            else:\n                st.error("Unable to verify that username.")\n        user = st.session_state.question_user\n        if user and store.security_question(user):\n            st.info(store.security_question(user))\n            with st.form("answer_question_form"):\n                answer = st.text_input("Your answer", type="password", max_chars=72)\n                verify = st.form_submit_button(\n                    "Verify answer", type="primary", width="stretch"\n                )\n            if verify:\n                if answer_error(answer):\n                    st.error(answer_error(answer))\n                else:\n                    grant = store.verify_answer(user, answer)\n                    if grant:\n                        st.session_state.reset_identity = store.user(user)["email"]\n                        st.session_state.reset_grant = grant\n                        st.session_state.reset_verified = True\n                        st.rerun()\n                    else:\n                        st.error("Incorrect answer or too many attempts.")\n    if st.button("Back to sign in"):\n        navigate("login")\n\n\nROLE_SUMMARIES = {\n    "customer": "Your account, activity and freight workspace.",\n    "operations_manager": "Your workspace for coordinating freight operations.",\n    "freight_broker": "Your workspace for coordinating quotes and carriers.",\n    "auditor": "Review account activity with read-only audit access.",\n    "admin": "Manage people, access and account security.",\n}\nROLE_MODULES = {\n    "customer": [\n        ("Shipment overview", "Track your shipments and delivery updates."),\n        ("Quotes", "Review freight options for your cargo."),\n    ],\n    "operations_manager": [\n        ("Command Center", "Coordinate operations, tickets and approvals."),\n        ("Route AI", "Review routes, fleet activity and weather risk."),\n    ],\n    "freight_broker": [\n        ("Spot Quotes", "Compare freight quotes and margins."),\n        ("Carriers", "Review carrier options and availability."),\n    ],\n    "auditor": [\n        ("Reports", "Review operational reports and decisions."),\n        ("Knowledge Graph", "Explore relationships across freight activity."),\n    ],\n    "admin": [\n        ("Platform overview", "Connect teams to the freight platform."),\n        ("Data Feed Center", "Review connected data sources."),\n    ],\n}\n\n\ndef event_table(events):\n    if not events:\n        st.info("No activity recorded yet.")\n        return\n    labels = {\n        "signup": "Account created",\n        "login_success": "Signed in",\n        "login_failed": "Sign-in unsuccessful",\n        "logout": "Signed out",\n        "email_verified": "Email verified",\n        "password_changed": "Password changed",\n        "account_locked_otp_failures": "Account locked · incorrect codes",\n        "account_locked_otp_budget": "Account locked · code limit",\n        "admin_login": "Administrator signed in",\n        "admin_login_failed": "Sign-in unsuccessful",\n    }\n    rows = []\n    for event in events:\n        raw = event["event"]\n        parts = raw.split(":")\n        if parts[0] == "role_changed" and len(parts) == 3:\n            text = f"Role changed · {parts[1]} → {ROLES.get(parts[2], parts[2])}"\n        elif parts[0] == "account_unlocked" and len(parts) == 2:\n            text = f"Account unlocked · {parts[1]}"\n        elif parts[0] == "otp_requested":\n            text = (\n                "Verification code requested"\n                if parts[-1] == "verify"\n                else "Recovery code requested"\n            )\n        else:\n            text = labels.get(raw, raw.replace("_", " ").capitalize())\n        row = {\n            "Activity": text,\n            "Time (UTC)": event["occurred_at"].replace("T", " ")[:19],\n        }\n        if "actor" in event:\n            row = {"Account": event["actor"], **row}\n        rows.append(row)\n    st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch")\n\n\ndef security_settings(account):\n    st.subheader("Account security")\n    if account["principal_type"] == "configured_admin":\n        st.info(\n            "The platform owner manages this administrator\'s credentials in the deployment settings."\n        )\n        return\n    st.caption("Changing your password ends all existing sessions.")\n    with st.form("change_password_form"):\n        current = st.text_input("Current password", type="password", max_chars=72)\n        new = st.text_input("New password", type="password", max_chars=72)\n        confirm = st.text_input("Confirm new password", type="password", max_chars=72)\n        st.caption("8+ characters: uppercase, lowercase, number and symbol. No spaces.")\n        save = st.form_submit_button("Update password", type="primary")\n    if save:\n        if not current:\n            st.error("Enter your current password.")\n        elif password_ok(new, confirm):\n            ok, message = store.change_password(account["email"], current, new)\n            if ok:\n                st.session_state.jwt = None\n                st.session_state.flash = message\n                navigate("login")\n            else:\n                st.error(message)\n\n\ndef manage_users(account):\n    users = store.visible_users(\n        st.session_state.jwt, JWT_SECRET, ADMIN_USERNAME, ADMIN_PASSWORD\n    )\n    for member in users:\n        store.is_locked(member["email"])\n    users = store.visible_users(\n        st.session_state.jwt, JWT_SECRET, ADMIN_USERNAME, ADMIN_PASSWORD\n    )\n    metrics = st.columns(3)\n    metrics[0].metric("Registered accounts", len(users))\n    metrics[1].metric("Verified", sum(bool(u["email_verified"]) for u in users))\n    metrics[2].metric("Locked", sum(bool(u["account_locked"]) for u in users))\n    directory, access = st.tabs(["Directory", "Access controls"])\n    with directory:\n        a, b = st.columns([2, 1])\n        query = a.text_input("Search users", placeholder="Username or email")\n        role_filter = b.selectbox("Filter by role", ["All roles", *ROLES.values()])\n        rows = [\n            {\n                "Username": u["username"],\n                "Email": u["email"],\n                "Role": ROLES.get(u["role"], "Unassigned"),\n                "Status": (\n                    "Locked"\n                    if u["account_locked"]\n                    else "Active" if u["email_verified"] else "Awaiting verification"\n                ),\n                "Last sign-in (UTC)": (\n                    u["last_login"].replace("T", " ")[:16] if u["last_login"] else "—"\n                ),\n            }\n            for u in users\n            if query.casefold() in (u["username"] + " " + u["email"]).casefold()\n            and (role_filter == "All roles" or ROLES.get(u["role"]) == role_filter)\n        ]\n        if rows:\n            st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch")\n        else:\n            st.info(\n                "No accounts match your search."\n                if users\n                else "No registered accounts yet. New accounts will appear here."\n            )\n    with access:\n        locked = [u["username"] for u in users if u["account_locked"]]\n        st.markdown("#### Unlock an account")\n        if locked:\n            selected = st.selectbox("Account to unlock", locked)\n            if st.button("Unlock account", type="primary"):\n                ok, message = store.unlock_account(\n                    selected,\n                    st.session_state.jwt,\n                    JWT_SECRET,\n                    ADMIN_USERNAME,\n                    ADMIN_PASSWORD,\n                )\n                if ok:\n                    st.session_state.flash = message\n                    st.rerun()\n                st.error(message)\n        else:\n            st.caption("No locked accounts.")\n        st.divider()\n        st.markdown("#### Assign a role")\n        eligible = [\n            u\n            for u in users\n            if u["email_verified"]\n            and not u["account_locked"]\n            and u["username"] != account["username"]\n        ]\n        if eligible:\n            names = {u["username"]: u for u in eligible}\n            selected = st.selectbox("Account", list(names), key="role_account")\n            current_role = names[selected]["role"]\n            st.caption("Current role: " + ROLES.get(current_role, "Unassigned"))\n            with st.form("assign_role_form"):\n                role = st.selectbox(\n                    "New role",\n                    list(ROLES),\n                    format_func=ROLES.get,\n                    index=None,\n                    placeholder="Choose a role",\n                )\n                confirmed = st.checkbox(\n                    "I confirm this account should receive the selected access."\n                )\n                save = st.form_submit_button("Save role", type="primary")\n            if save:\n                if not role or not confirmed:\n                    st.error("Choose a role and confirm the access change.")\n                else:\n                    ok, message = store.assign_role(\n                        selected,\n                        role,\n                        st.session_state.jwt,\n                        JWT_SECRET,\n                        ADMIN_USERNAME,\n                        ADMIN_PASSWORD,\n                    )\n                    if ok:\n                        st.session_state.flash = message\n                        st.rerun()\n                    st.error(message)\n            st.caption(\n                "Role changes end the user\'s existing sessions. Admins can manage all accounts; Auditors can read the audit trail."\n            )\n        else:\n            st.caption(\n                "A verified, unlocked account is required. You cannot change your own role."\n            )\n\n\ndef workspace(account):\n    # Revalidate every rerun; the sidebar is presentation, never the permission check.\n    account = current_user()\n    if not account:\n        navigate("login")\n    role_label = ROLES[account["role"]]\n    items = ["Workspace", "My profile", "Security"]\n    if can(account, "users"):\n        items.append("User management")\n    if can(account, "audit"):\n        items.append("Audit trail")\n    with st.sidebar:\n        st.markdown("### Your workspace")\n        st.caption(role_label)\n        if st.session_state.get("workspace_page") not in items:\n            st.session_state.workspace_page = "Workspace"\n        selected = st.radio(\n            "Navigation", items, key="workspace_page", label_visibility="collapsed"\n        )\n        st.divider()\n        st.write(account["username"])\n        st.caption("Signed in")\n        if st.button("Log out", width="stretch"):\n            log_out()\n    flash()\n    st.markdown(\n        f\'<div class="workspace-heading"><div><div class="eyebrow">FreightQuote AI / WORKSPACE</div><h2>{html.escape(selected)}</h2></div><span class="role-badge">{html.escape(role_label)}</span></div>\',\n        unsafe_allow_html=True,\n    )\n    if selected == "Workspace":\n        st.markdown(\n            f\'<section class="welcome-panel"><div class="eyebrow">WELCOME BACK</div><h2>Hello, {html.escape(account["username"])}.</h2><p>{ROLE_SUMMARIES[account["role"]]}</p></section>\',\n            unsafe_allow_html=True,\n        )\n        cols = st.columns(3)\n        cols[0].metric("Your role", role_label)\n        cols[1].metric("Account", "Active")\n        payload = decode_token(st.session_state.jwt, JWT_SECRET)\n        cols[2].metric(\n            "Session remaining",\n            str(max(0, (payload["exp"] - int(time.time())) // 60)) + " min",\n        )\n        st.markdown("#### Freight workspace")\n        st.caption("The following modules are planned for dashboard integration.")\n        cards = "".join(\n            f\'<article class="module-card"><span class="module-label">PLANNED</span><h4>{name}</h4><p>{description}</p></article>\'\n            for name, description in ROLE_MODULES[account["role"]]\n        )\n        st.markdown(\n            \'<div class="module-grid">\' + cards + "</div>", unsafe_allow_html=True\n        )\n        st.markdown("#### Recent account activity")\n        event_table(store.account_events(account["username"], 8))\n    elif selected == "My profile":\n        with st.container(border=True):\n            st.write("**Username:**", account["username"])\n            st.write("**Role:**", role_label)\n            if account["email"]:\n                st.write("**Email:**", account["email"])\n                st.write("**Email status:** Verified")\n                st.write("**Member since:**", account["created_at"][:10])\n                st.write("**Successful sign-ins:**", account["login_count"])\n            else:\n                st.caption(\n                    "Platform administrator · configured by the deployment owner."\n                )\n        st.caption("Contact an administrator if your role needs to change.")\n    elif selected == "Security":\n        security_settings(account)\n    elif selected == "User management" and can(account, "users"):\n        manage_users(account)\n    elif selected == "Audit trail" and can(account, "audit"):\n        st.caption("Latest 100 account events. Read-only; all times are UTC.")\n        event_table(\n            store.audit_events(\n                st.session_state.jwt, JWT_SECRET, ADMIN_USERNAME, ADMIN_PASSWORD\n            )\n        )\n\n\n@st.fragment(run_every="5s")\ndef session_watch():\n    if st.session_state.jwt and not current_user():\n        st.session_state.jwt = None\n        st.session_state.flash = (\n            "Session ended. Sign in again or contact the administrator."\n        )\n        navigate("login")\n\n\nif len(JWT_SECRET) < 32:\n    st.error(\n        "JWT signing key is not configured. Run the Colab configuration cell first."\n    )\n    st.stop()\n\ntopbar()\nviewer = current_user()\nif viewer:\n    session_watch()\nif viewer:\n    if st.session_state.view != "dashboard":\n        navigate("dashboard")\n    workspace(viewer)\nelse:\n    if st.session_state.jwt:\n        st.session_state.jwt = None\n    with st.container(key="auth_shell"):\n        left, right = st.columns([1.12, 1], gap="small", vertical_alignment="center")\n        with left:\n            left_panel()\n        with right, st.container(key="auth_glass"):\n            route = st.session_state.view\n            if route == "locked":\n                locked_page()\n            elif route == "signup":\n                signup()\n            elif route == "forgot":\n                forgot()\n            elif route == "verify-email":\n                verify_email_page()\n            else:\n                login()\n\nbuild_label = (\n    " · <strong>Build: FreightQuote AI-AUTH-03</strong>"\n    if os.getenv("SHOW_BUILD_ID", "0") == "1"\n    else ""\n)\nst.markdown(\n    \'<div class="footer-note">FreightQuote AI · Freight intelligence\'\n    + build_label\n    + "</div>",\n    unsafe_allow_html=True,\n)\n', 'auth_core.py': '"""FreightQuote AI account store, JWT sessions and email verification.\n\nSQLite is used for the single-instance milestone build. Never store plaintext\npasswords, OTPs, recovery answers or secret values in the database.\n"""\n\nimport datetime as dt\nimport hashlib\nimport hmac\nimport re\nimport secrets\nimport sqlite3\nimport time\nimport unicodedata\nfrom contextlib import contextmanager\nfrom pathlib import Path\n\nimport bcrypt\nimport jwt\n\nEMAIL_RE = re.compile(\n    r"^[A-Za-z]{2,}[A-Za-z0-9._%+\\-]*@[A-Za-z]{2,}[A-Za-z0-9\\-]*(?:\\.[A-Za-z]{2,}[A-Za-z0-9\\-]*)+$"\n)\nQUESTIONS = [\n    "What was the name of your first school?",\n    "In which city were you born?",\n    "What was the name of your first pet?",\n    "What is the middle name of your oldest sibling?",\n]\nGENERIC_LOGIN_ERROR = "Invalid username/email or password."\nUTC = dt.timezone.utc\nLOCKED_MESSAGE = "Account locked. Contact the administrator to unlock it."\nOTP_TTL = 300\nMAX_OTP_REQUESTS = 5\nMAX_OTP_FAILURES = 5\n\nROLES = {\n    "customer": "Customer",\n    "operations_manager": "Operations Manager",\n    "freight_broker": "Freight Broker",\n    "auditor": "Auditor",\n    "admin": "Admin",\n}\nROLE_PERMISSIONS = {\n    role: frozenset(\n        {"profile", "security", "workspace"}\n        | ({"users", "manage_access", "audit"} if role == "admin" else set())\n        | ({"audit"} if role == "auditor" else set())\n    )\n    for role in ROLES\n}\n\n\ndef can(account, permission):\n    return bool(account and permission in ROLE_PERMISSIONS.get(account.get("role"), ()))\n\n\ndef admin_credential_tag(password, secret):\n    return hmac.new(\n        secret.encode(), ("admin:" + password).encode(), hashlib.sha256\n    ).hexdigest()\n\n\ndef clean_answer(value):\n    """Ignore case and surrounding whitespace; reject invisible-only answers."""\n    value = unicodedata.normalize("NFKC", value)\n    value = "".join(ch for ch in value if unicodedata.category(ch) != "Cf")\n    return value.strip().casefold()\n\n\ndef answer_error(value):\n    cleaned = clean_answer(value)\n    if not cleaned or not any(ch.isalnum() for ch in cleaned):\n        return "Enter a security answer containing a letter or number."\n    if len(cleaned.encode("utf-8")) > 72:\n        return "Security answer is too long (maximum 72 UTF-8 bytes)."\n    return None\n\n\ndef utcnow():\n    return dt.datetime.now(UTC).isoformat()\n\n\ndef valid_email(value):\n    return (\n        isinstance(value, str)\n        and len(value.strip()) <= 254\n        and bool(EMAIL_RE.fullmatch(value.strip()))\n    )\n\n\ndef password_errors(password):\n    issues = []\n    if len(password) < 8:\n        issues.append("at least 8 characters")\n    if len(password.encode("utf-8")) > 72:\n        issues.append("no more than 72 UTF-8 bytes")\n    if not re.search(r"[A-Z]", password):\n        issues.append("one uppercase letter")\n    if not re.search(r"[a-z]", password):\n        issues.append("one lowercase letter")\n    if not re.search(r"[0-9]", password):\n        issues.append("one number")\n    if not any(\n        not ch.isalnum()\n        and not ch.isspace()\n        and not unicodedata.category(ch).startswith("C")\n        for ch in password\n    ):\n        issues.append("one special symbol (spaces do not count)")\n    if any(ch.isspace() or unicodedata.category(ch).startswith("C") for ch in password):\n        issues.append("no whitespace or invisible characters")\n    return issues\n\n\ndef hashed(text):\n    return bcrypt.hashpw(text.encode("utf-8"), bcrypt.gensalt()).decode("ascii")\n\n\ndef matches(text, saved):\n    try:\n        if len(text.encode("utf-8")) > 72:\n            return False\n        return bcrypt.checkpw(text.encode("utf-8"), saved.encode("ascii"))\n    except (ValueError, TypeError, AttributeError):\n        return False\n\n\ndef otp_digest(code, salt, secret, purpose="reset"):\n    return hmac.new(\n        secret.encode(), f"{purpose}:{salt}:{code}".encode(), hashlib.sha256\n    ).hexdigest()\n\n\nclass Store:\n    def __init__(self, path="auth.db"):\n        self.path = str(path)\n        Path(self.path).parent.mkdir(parents=True, exist_ok=True)\n        self.init()\n\n    @contextmanager\n    def db(self):\n        conn = sqlite3.connect(self.path, timeout=20)\n        conn.row_factory = sqlite3.Row\n        try:\n            conn.execute("PRAGMA foreign_keys=ON")\n            conn.execute("PRAGMA busy_timeout=20000")\n            yield conn\n            conn.commit()\n        except Exception:\n            conn.rollback()\n            raise\n        finally:\n            conn.close()\n\n    def init(self):\n        with self.db() as c:\n            c.execute("""CREATE TABLE IF NOT EXISTS users (\n                id INTEGER PRIMARY KEY AUTOINCREMENT,\n                username TEXT NOT NULL UNIQUE COLLATE NOCASE,\n                email TEXT NOT NULL UNIQUE COLLATE NOCASE,\n                password_hash TEXT NOT NULL,\n                security_question TEXT NOT NULL,\n                answer_hash TEXT NOT NULL,\n                created_at TEXT NOT NULL, updated_at TEXT NOT NULL,\n                login_count INTEGER DEFAULT 0, last_login TEXT,\n                previous_login TEXT, session_version INTEGER DEFAULT 0,\n                email_verified INTEGER DEFAULT 1\n            )""")\n            # Upgrade existing milestone databases without dropping accounts.\n            cols = {r["name"] for r in c.execute("PRAGMA table_info(users)")}\n            for name, ddl in (\n                ("previous_login", "TEXT"),\n                ("session_version", "INTEGER NOT NULL DEFAULT 0"),\n                ("email_verified", "INTEGER NOT NULL DEFAULT 1"),\n                ("account_locked", "INTEGER NOT NULL DEFAULT 0"),\n                ("otp_failures", "INTEGER NOT NULL DEFAULT 0"),\n                ("role", "TEXT NOT NULL DEFAULT \'customer\'"),\n            ):\n                if name not in cols:\n                    c.execute(f"ALTER TABLE users ADD COLUMN {name} {ddl}")\n            c.execute("""CREATE TABLE IF NOT EXISTS resets (\n                email TEXT PRIMARY KEY COLLATE NOCASE, code_hash TEXT NOT NULL,\n                salt TEXT NOT NULL, expires_at INTEGER NOT NULL,\n                attempts INTEGER NOT NULL DEFAULT 0, sent_at INTEGER NOT NULL,\n                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE\n            )""")\n            c.execute("""CREATE TABLE IF NOT EXISTS email_verifications (\n                email TEXT PRIMARY KEY COLLATE NOCASE, code_hash TEXT NOT NULL,\n                salt TEXT NOT NULL, expires_at INTEGER NOT NULL,\n                attempts INTEGER NOT NULL DEFAULT 0, sent_at INTEGER NOT NULL,\n                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE\n            )""")\n            c.execute("""CREATE TABLE IF NOT EXISTS code_dispatch (\n                email TEXT NOT NULL COLLATE NOCASE, purpose TEXT NOT NULL,\n                sent_at INTEGER NOT NULL, PRIMARY KEY(email,purpose)\n            )""")\n            c.execute("""CREATE TABLE IF NOT EXISTS limits (\n                subject TEXT PRIMARY KEY, attempts INTEGER NOT NULL,\n                window_start INTEGER NOT NULL, locked_until INTEGER NOT NULL DEFAULT 0\n            )""")\n            c.execute("""CREATE TABLE IF NOT EXISTS audit (\n                id INTEGER PRIMARY KEY AUTOINCREMENT, actor TEXT NOT NULL,\n                event TEXT NOT NULL, occurred_at TEXT NOT NULL\n            )""")\n            c.execute("""CREATE TABLE IF NOT EXISTS otp_policy (\n                email TEXT NOT NULL COLLATE NOCASE, purpose TEXT NOT NULL,\n                requests INTEGER NOT NULL DEFAULT 0, next_request_at INTEGER NOT NULL DEFAULT 0,\n                PRIMARY KEY(email,purpose),\n                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE\n            )""")\n            c.execute("""CREATE TABLE IF NOT EXISTS reset_grants (\n                email TEXT PRIMARY KEY COLLATE NOCASE, token_hash TEXT NOT NULL,\n                expires_at INTEGER NOT NULL, version INTEGER NOT NULL,\n                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE\n            )""")\n            c.execute(\n                "CREATE INDEX IF NOT EXISTS audit_actor_id ON audit(actor,id DESC)"\n            )\n\n    def audit(self, actor, event):\n        with self.db() as c:\n            c.execute(\n                "INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)",\n                (str(actor)[:100], event, utcnow()),\n            )\n\n    def limited(self, subject):\n        with self.db() as c:\n            r = c.execute(\n                "SELECT locked_until FROM limits WHERE subject=?", (subject,)\n            ).fetchone()\n            return bool(r and r["locked_until"] > int(time.time()))\n\n    def failure(self, subject, maximum=5, period=900):\n        now = int(time.time())\n        with self.db() as c:\n            row = c.execute(\n                "SELECT * FROM limits WHERE subject=?", (subject,)\n            ).fetchone()\n            if not row or now - row["window_start"] >= period:\n                c.execute(\n                    "INSERT OR REPLACE INTO limits VALUES(?,?,?,?)",\n                    (subject, 1, now, 0),\n                )\n            else:\n                n = row["attempts"] + 1\n                c.execute(\n                    "UPDATE limits SET attempts=?, locked_until=? WHERE subject=?",\n                    (n, now + period if n >= maximum else 0, subject),\n                )\n\n    def clear_limit(self, subject):\n        with self.db() as c:\n            c.execute("DELETE FROM limits WHERE subject=?", (subject,))\n\n    def signup(\n        self,\n        username,\n        email,\n        password,\n        question,\n        answer,\n        *,\n        require_verification=False,\n        reserved_identity="admin",\n    ):\n        username = username.strip()\n        email = email.strip().lower()\n        if reserved_identity and reserved_identity.strip().casefold() in (\n            username.casefold(),\n            email.casefold(),\n        ):\n            return (\n                False,\n                "That identity is reserved. Choose another username and email.",\n            )\n        if not re.fullmatch(r"[\\w.\\-]{3,50}", username, re.UNICODE):\n            return (\n                False,\n                "Username must be 3–50 letters/digits/dots/dashes/underscores.",\n            )\n        if not valid_email(email):\n            return False, "Enter a valid email address."\n        if password_errors(password):\n            return False, "Password does not meet the requirements."\n        if question not in QUESTIONS:\n            return False, "Choose a security question."\n        if answer_error(answer):\n            return False, answer_error(answer)\n        now = utcnow()\n        try:\n            with self.db() as c:\n                c.execute(\n                    """INSERT INTO users(username,email,password_hash,security_question,\n                    answer_hash,created_at,updated_at,email_verified)\n                    VALUES(?,?,?,?,?,?,?,?)""",\n                    (\n                        username,\n                        email,\n                        hashed(password),\n                        question,\n                        hashed(clean_answer(answer)),\n                        now,\n                        now,\n                        0 if require_verification else 1,\n                    ),\n                )\n        except sqlite3.IntegrityError:\n            with self.db() as c:\n                name_taken = c.execute(\n                    "SELECT 1 FROM users WHERE username=?", (username,)\n                ).fetchone()\n            return False, (\n                "Username already exists."\n                if name_taken\n                else "Email address already registered."\n            )\n        self.audit(username, "signup")\n        return True, (\n            "Account created. Verify your email to activate it."\n            if require_verification\n            else "Account created. Sign in to continue."\n        )\n\n    def user(self, identity):\n        identity = str(identity).strip()\n        with self.db() as c:\n            return c.execute(\n                "SELECT * FROM users WHERE username=? OR email=?", (identity, identity)\n            ).fetchone()\n\n    def authenticate(self, identity, password):\n        row = self.user(identity)\n        key = "login:" + (str(row["id"]) if row else identity.strip().casefold())\n        if self.limited(key):\n            return None, "Too many attempts. Try again in 15 minutes."\n        if not row or not matches(password, row["password_hash"]):\n            self.failure(key)\n            self.audit(identity[:100], "login_failed")\n            return None, GENERIC_LOGIN_ERROR\n        if self.is_locked(row["email"]):\n            return None, LOCKED_MESSAGE\n        if not row["email_verified"]:\n            return None, "Please verify your registered email before signing in."\n        self.clear_limit(key)\n        now = utcnow()\n        with self.db() as c:\n            c.execute(\n                """UPDATE users SET previous_login=last_login,last_login=?,login_count=login_count+1\n                         WHERE id=?""",\n                (now, row["id"]),\n            )\n        self.audit(row["username"], "login_success")\n        return self.user(identity), None\n\n    def sign_in(self, identity, password, admin_username="admin", admin_password=""):\n        """One entry point for configured administrators and registered accounts."""\n        identity = identity.strip()\n        if identity.casefold() == admin_username.strip().casefold():\n            if self.limited("admin-login"):\n                return None, "Too many attempts. Try again in 15 minutes."\n            if not admin_password or not hmac.compare_digest(\n                password.encode(), admin_password.encode()\n            ):\n                self.failure("admin-login")\n                self.audit(admin_username, "login_failed")\n                return None, GENERIC_LOGIN_ERROR\n            self.clear_limit("admin-login")\n            self.audit(admin_username, "login_success")\n            return {\n                "username": admin_username,\n                "email": None,\n                "role": "admin",\n                "principal_type": "configured_admin",\n                "session_version": 0,\n            }, None\n        account, error = self.authenticate(identity, password)\n        if error:\n            return None, error\n        if account["role"] not in ROLES:\n            return None, GENERIC_LOGIN_ERROR\n        return dict(account) | {"principal_type": "account"}, None\n\n    def session(self, token, secret, admin_username="admin", admin_password=None):\n        payload = decode_token(token, secret) if token and len(secret) >= 32 else None\n        if not payload:\n            return None\n        if payload.get("kind") == "configured_admin":\n            if payload.get("sub") != admin_username or payload.get("role") != "admin":\n                return None\n            if admin_password is not None and (\n                not admin_password\n                or not hmac.compare_digest(\n                    payload.get("credential_tag", ""),\n                    admin_credential_tag(admin_password, secret),\n                )\n            ):\n                return None\n            return {\n                "username": admin_username,\n                "email": None,\n                "role": "admin",\n                "principal_type": "configured_admin",\n                "session_version": 0,\n            }\n        if payload.get("kind") != "account":\n            return None\n        row = self.user(payload.get("sub", ""))\n        if not row or self.is_locked(row["email"]):\n            return None\n        row = self.user(row["email"])\n        if (\n            not row["email_verified"]\n            or row["role"] not in ROLES\n            or row["role"] != payload.get("role")\n            or row["session_version"] != payload.get("ver")\n        ):\n            return None\n        return dict(row) | {"principal_type": "account"}\n\n    def _still_authorized(self, conn, account, permission):\n        """Recheck stored permissions inside the same transaction as a mutation."""\n        if not can(account, permission):\n            return False\n        if account.get("principal_type") == "configured_admin":\n            return True\n        row = conn.execute(\n            "SELECT * FROM users WHERE id=?", (account["id"],)\n        ).fetchone()\n        return bool(\n            row\n            and not row["account_locked"]\n            and row["email_verified"]\n            and row["role"] == account["role"]\n            and row["session_version"] == account["session_version"]\n        )\n\n    def assign_role(\n        self,\n        identity,\n        role,\n        admin_token,\n        secret,\n        admin_username="admin",\n        admin_password=None,\n    ):\n        account = self.session(admin_token, secret, admin_username, admin_password)\n        if not can(account, "manage_access"):\n            return False, "Administrator authentication required."\n        if role not in ROLES:\n            return False, "Choose a valid role."\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            if not self._still_authorized(c, account, "manage_access"):\n                return False, "Your access changed. Sign in again."\n            row = c.execute(\n                "SELECT * FROM users WHERE username=? OR email=?", (identity, identity)\n            ).fetchone()\n            if not row:\n                return False, "Account not found."\n            if account.get("id") == row["id"]:\n                return False, "Ask another administrator to change your role."\n            if row["account_locked"] or not row["email_verified"]:\n                return False, "Verify and unlock the account before changing its role."\n            if row["role"] == role:\n                return False, "This account already has that role."\n            c.execute(\n                "UPDATE users SET role=?,session_version=session_version+1,updated_at=? WHERE id=?",\n                (role, utcnow(), row["id"]),\n            )\n            c.execute("DELETE FROM reset_grants WHERE email=?", (row["email"],))\n            c.execute(\n                "INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)",\n                (\n                    account["username"],\n                    "role_changed:" + row["username"] + ":" + role,\n                    utcnow(),\n                ),\n            )\n        return True, "Role updated. The user must sign in again."\n\n    def visible_users(self, token, secret, admin_username="admin", admin_password=None):\n        account = self.session(token, secret, admin_username, admin_password)\n        if not can(account, "users"):\n            raise PermissionError("Administrator authentication required.")\n        return self.list_users()\n\n    def audit_events(self, token, secret, admin_username="admin", admin_password=None):\n        account = self.session(token, secret, admin_username, admin_password)\n        if not can(account, "audit"):\n            raise PermissionError("Audit access required.")\n        return self.recent_events(100)\n\n    def _lock(self, conn, row, reason):\n        conn.execute(\n            """UPDATE users SET account_locked=1,\n                     session_version=session_version+1 WHERE id=? AND account_locked=0""",\n            (row["id"],),\n        )\n        for table in ("resets", "email_verifications", "reset_grants"):\n            conn.execute(f"DELETE FROM {table} WHERE email=?", (row["email"],))\n        conn.execute(\n            "INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)",\n            (row["username"], reason, utcnow()),\n        )\n\n    def is_locked(self, identity):\n        row = self.user(identity)\n        if not row:\n            return False\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            row = c.execute("SELECT * FROM users WHERE id=?", (row["id"],)).fetchone()\n            if row["account_locked"]:\n                return True\n            # The last allowed code remains usable for its entire validity period.\n            for purpose, table in (\n                ("reset", "resets"),\n                ("verify", "email_verifications"),\n            ):\n                expired = c.execute(\n                    f"""SELECT 1 FROM otp_policy p JOIN {table} r\n                    ON p.email=r.email WHERE p.email=? AND p.purpose=?\n                    AND p.requests>=? AND r.expires_at<=?""",\n                    (row["email"], purpose, MAX_OTP_REQUESTS, int(time.time())),\n                ).fetchone()\n                if expired:\n                    self._lock(c, row, "account_locked_otp_budget")\n                    return True\n        return False\n\n    def unlock_account(\n        self, identity, admin_token, secret, admin_username="admin", admin_password=None\n    ):\n        account = self.session(admin_token, secret, admin_username, admin_password)\n        if not can(account, "manage_access"):\n            return False, "Administrator authentication required."\n        row = self.user(identity)\n        if not row:\n            return False, "Account not found."\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            if not self._still_authorized(c, account, "manage_access"):\n                return False, "Your access changed. Sign in again."\n            c.execute(\n                """UPDATE users SET account_locked=0,otp_failures=0,\n                         session_version=session_version+1 WHERE id=?""",\n                (row["id"],),\n            )\n            for table in (\n                "resets",\n                "email_verifications",\n                "reset_grants",\n                "otp_policy",\n                "code_dispatch",\n            ):\n                c.execute(f"DELETE FROM {table} WHERE email=?", (row["email"],))\n            for subject in (\n                "login:" + str(row["id"]),\n                "login:" + row["username"].casefold(),\n                "login:" + row["email"].casefold(),\n                "question:" + row["username"].casefold(),\n            ):\n                c.execute("DELETE FROM limits WHERE subject=?", (subject,))\n            c.execute(\n                "INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)",\n                (account["username"], "account_unlocked:" + row["username"], utcnow()),\n            )\n        return True, "Account unlocked. The user can request a new code."\n\n    def _new_grant(self, conn, row):\n        token = secrets.token_urlsafe(32)\n        conn.execute(\n            "INSERT OR REPLACE INTO reset_grants VALUES(?,?,?,?)",\n            (\n                row["email"],\n                hashlib.sha256(token.encode()).hexdigest(),\n                int(time.time()) + 300,\n                row["session_version"],\n            ),\n        )\n        return token\n\n    def change_password(self, identity, current, new):\n        if password_errors(new):\n            return False, "Password does not meet the requirements."\n        row = self.user(identity)\n        if not row or not matches(current, row["password_hash"]):\n            return False, "Current password is incorrect."\n        if self.is_locked(identity):\n            return False, LOCKED_MESSAGE\n        if matches(new, row["password_hash"]):\n            return False, "Choose a password different from your current password."\n        new_hash = hashed(new)\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            current_row = c.execute(\n                "SELECT * FROM users WHERE id=?", (row["id"],)\n            ).fetchone()\n            if (\n                current_row["account_locked"]\n                or current_row["session_version"] != row["session_version"]\n            ):\n                return False, "Session changed. Sign in again."\n            self._set_password(c, current_row, new_hash)\n        return True, "Password updated. Sign in again."\n\n    def _set_password(self, conn, row, new_hash):\n        conn.execute(\n            """UPDATE users SET password_hash=?,updated_at=?,\n            session_version=session_version+1,otp_failures=0 WHERE id=?""",\n            (new_hash, utcnow(), row["id"]),\n        )\n        for table in (\n            "resets",\n            "email_verifications",\n            "reset_grants",\n            "otp_policy",\n            "code_dispatch",\n        ):\n            conn.execute(f"DELETE FROM {table} WHERE email=?", (row["email"],))\n        conn.execute(\n            "INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)",\n            (row["username"], "password_changed", utcnow()),\n        )\n\n    def reset_password(self, email, new_password, grant=None):\n        if password_errors(new_password):\n            return False, "Password does not meet the requirements."\n        row = self.user(email)\n        if not row or not grant:\n            return False, "Verify your identity again."\n        if self.is_locked(email):\n            return False, LOCKED_MESSAGE\n        new_hash = hashed(new_password)\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            row = c.execute("SELECT * FROM users WHERE id=?", (row["id"],)).fetchone()\n            saved = c.execute(\n                "SELECT * FROM reset_grants WHERE email=?", (row["email"],)\n            ).fetchone()\n            if row["account_locked"]:\n                return False, LOCKED_MESSAGE\n            if (\n                not saved\n                or saved["expires_at"] <= int(time.time())\n                or saved["version"] != row["session_version"]\n                or not hmac.compare_digest(\n                    saved["token_hash"], hashlib.sha256(grant.encode()).hexdigest()\n                )\n            ):\n                return False, "Verification expired. Verify your identity again."\n            if matches(new_password, row["password_hash"]):\n                return False, "Choose a password different from your current password."\n            self._set_password(c, row, new_hash)\n        return True, "Password updated. Sign in again."\n\n    def security_question(self, username):\n        row = self.user(username)\n        if not row or self.is_locked(username):\n            return None\n        return (\n            row["security_question"]\n            if row["username"].casefold() == username.strip().casefold()\n            else None\n        )\n\n    def verify_answer(self, username, answer):\n        if answer_error(answer):\n            return False\n        key = "question:" + username.strip().casefold()\n        if self.limited(key) or self.is_locked(username):\n            return False\n        row = self.user(username)\n        ok = bool(\n            row\n            and row["username"].casefold() == username.strip().casefold()\n            and matches(clean_answer(answer), row["answer_hash"])\n        )\n        if not ok:\n            self.failure(key)\n            return False\n        self.clear_limit(key)\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            row = c.execute("SELECT * FROM users WHERE id=?", (row["id"],)).fetchone()\n            if row["account_locked"]:\n                return False\n            return self._new_grant(c, row)\n\n    def code_status(self, email, purpose="reset"):\n        row = self.user(email)\n        empty = {\n            "locked": False,\n            "requests": 0,\n            "failures": 0,\n            "resend_in": 0,\n            "expires_in": 0,\n        }\n        if not row:\n            return empty\n        locked = self.is_locked(email)\n        row = self.user(email)\n        table = "resets" if purpose == "reset" else "email_verifications"\n        with self.db() as c:\n            policy = c.execute(\n                "SELECT * FROM otp_policy WHERE email=? AND purpose=?",\n                (row["email"], purpose),\n            ).fetchone()\n            code = c.execute(\n                f"SELECT expires_at FROM {table} WHERE email=?", (row["email"],)\n            ).fetchone()\n        now = int(time.time())\n        return {\n            "locked": locked,\n            "requests": policy["requests"] if policy else 0,\n            "failures": row["otp_failures"],\n            "resend_in": max(0, policy["next_request_at"] - now) if policy else 0,\n            "expires_in": max(0, code["expires_at"] - now) if code else 0,\n        }\n\n    def _begin_code(self, email, secret, table, purpose):\n        if not valid_email(email) or len(secret) < 32:\n            return None, "Enter a valid email address."\n        row = self.user(email)\n        if not row or (purpose == "verify" and row["email_verified"]):\n            return None, "If this address is eligible, a code will be sent."\n        if self.is_locked(email):\n            return None, LOCKED_MESSAGE\n        now = int(time.time())\n        code = f"{secrets.randbelow(1000000):06d}"\n        salt = secrets.token_hex(16)\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            row = c.execute("SELECT * FROM users WHERE id=?", (row["id"],)).fetchone()\n            if row["account_locked"]:\n                return None, LOCKED_MESSAGE\n            old = c.execute(\n                "SELECT * FROM otp_policy WHERE email=? AND purpose=?",\n                (row["email"], purpose),\n            ).fetchone()\n            if old and old["next_request_at"] > now:\n                return (\n                    None,\n                    f"Request another code in {old[\'next_request_at\'] - now} seconds.",\n                )\n            if old and old["requests"] >= MAX_OTP_REQUESTS:\n                self._lock(c, row, "account_locked_otp_budget")\n                return None, LOCKED_MESSAGE\n            number = (old["requests"] if old else 0) + 1\n            delay = 60 if number <= 3 else 300\n            c.execute(\n                "INSERT OR REPLACE INTO otp_policy VALUES(?,?,?,?)",\n                (row["email"], purpose, number, now + delay),\n            )\n            c.execute(\n                f"""INSERT OR REPLACE INTO {table}\n                (email,code_hash,salt,expires_at,attempts,sent_at) VALUES(?,?,?,?,0,?)""",\n                (\n                    row["email"],\n                    otp_digest(code, salt, secret, purpose),\n                    salt,\n                    now + OTP_TTL,\n                    now,\n                ),\n            )\n            c.execute("DELETE FROM reset_grants WHERE email=?", (row["email"],))\n            c.execute(\n                "INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)",\n                (row["username"], "otp_requested:" + purpose, utcnow()),\n            )\n        return code, None\n\n    def begin_otp(self, email, secret):\n        return self._begin_code(email, secret, "resets", "reset")\n\n    def begin_email_verification(self, email, secret):\n        return self._begin_code(email, secret, "email_verifications", "verify")\n\n    def cancel_otp(self, email):\n        self._cancel_code(email, "resets")\n\n    def cancel_email_verification(self, email):\n        self._cancel_code(email, "email_verifications")\n\n    def _cancel_code(self, email, table):\n        # Retain rate limits when SMTP fails; failures must not bypass cooldowns.\n        row = self.user(email)\n        if row:\n            with self.db() as c:\n                c.execute(f"DELETE FROM {table} WHERE email=?", (row["email"],))\n\n    def _verify_code(self, email, code, secret, table, purpose):\n        row = self.user(email)\n        if not row or not str(code).strip() or self.is_locked(email):\n            return False\n        with self.db() as c:\n            c.execute("BEGIN IMMEDIATE")\n            row = c.execute("SELECT * FROM users WHERE id=?", (row["id"],)).fetchone()\n            if row["account_locked"]:\n                return False\n            rec = c.execute(\n                f"SELECT * FROM {table} WHERE email=?", (row["email"],)\n            ).fetchone()\n            if not rec or rec["expires_at"] <= int(time.time()):\n                return False\n            ok = bool(re.fullmatch(r"[0-9]{6}", str(code))) and hmac.compare_digest(\n                otp_digest(str(code), rec["salt"], secret, purpose), rec["code_hash"]\n            )\n            if not ok:\n                failures = row["otp_failures"] + 1\n                c.execute(\n                    "UPDATE users SET otp_failures=? WHERE id=?", (failures, row["id"])\n                )\n                c.execute(\n                    f"UPDATE {table} SET attempts=attempts+1 WHERE email=?",\n                    (row["email"],),\n                )\n                if failures >= MAX_OTP_FAILURES:\n                    self._lock(c, row, "account_locked_otp_failures")\n                return False\n            c.execute(f"DELETE FROM {table} WHERE email=?", (row["email"],))\n            if purpose == "verify":\n                c.execute(\n                    "UPDATE users SET email_verified=1,otp_failures=0,updated_at=? WHERE id=?",\n                    (utcnow(), row["id"]),\n                )\n                c.execute(\n                    "DELETE FROM otp_policy WHERE email=? AND purpose=\'verify\'",\n                    (row["email"],),\n                )\n                c.execute(\n                    "INSERT INTO audit(actor,event,occurred_at) VALUES(?,?,?)",\n                    (row["username"], "email_verified", utcnow()),\n                )\n                return True\n            return self._new_grant(c, row)\n\n    def verify_otp(self, email, code, secret):\n        """Return a short-lived reset grant on success, otherwise False."""\n        return self._verify_code(email, code, secret, "resets", "reset")\n\n    def verify_email(self, email, code, secret):\n        return self._verify_code(email, code, secret, "email_verifications", "verify")\n\n    def recent_events(self, limit=50):\n        with self.db() as c:\n            return [\n                dict(x)\n                for x in c.execute(\n                    "SELECT actor,event,occurred_at FROM audit ORDER BY id DESC LIMIT ?",\n                    (limit,),\n                )\n            ]\n\n    def account_events(self, username, limit=30):\n        with self.db() as c:\n            return [\n                dict(x)\n                for x in c.execute(\n                    """SELECT event,occurred_at FROM audit WHERE actor=? COLLATE NOCASE\n                                              ORDER BY id DESC LIMIT ?""",\n                    (username, limit),\n                )\n            ]\n\n    def list_users(self):\n        with self.db() as c:\n            return [\n                dict(x)\n                for x in c.execute(\n                    """SELECT id,username,email,role,email_verified,account_locked,otp_failures,created_at,\n                previous_login,last_login,login_count FROM users ORDER BY id DESC"""\n                )\n            ]\n\n    def delete_user(self, user_id):\n        with self.db() as c:\n            c.execute("DELETE FROM users WHERE id=?", (user_id,))\n\n\ndef issue_token(\n    subject,\n    secret,\n    role="customer",\n    lifetime=7200,\n    version=0,\n    kind="account",\n    credential_tag="",\n):\n    now = dt.datetime.now(UTC)\n    return jwt.encode(\n        {\n            "sub": str(subject),\n            "role": role,\n            "kind": kind,\n            "credential_tag": credential_tag,\n            "ver": int(version),\n            "iat": now,\n            "exp": now + dt.timedelta(seconds=lifetime),\n            "jti": secrets.token_hex(16),\n        },\n        secret,\n        algorithm="HS256",\n    )\n\n\ndef decode_token(token, secret):\n    try:\n        return jwt.decode(\n            token,\n            secret,\n            algorithms=["HS256"],\n            options={"require": ["sub", "exp", "iat"]},\n        )\n    except (jwt.InvalidTokenError, TypeError, ValueError):\n        return None\n', 'motion.css': '/* PORTWISE presentation. All motion respects reduced-motion and Pause motion. */\nhtml, body, [data-testid="stApp"] {\n  font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;\n  color: var(--pw-ink);\n}\n[data-testid="stApp"] { background: var(--pw-bg); }\n[data-testid="stHeader"] { background: transparent; }\n.block-container { max-width: 1280px; padding: 2rem 2.6rem 1rem; }\nh1,h2,h3,h4 { color: var(--pw-ink)!important; letter-spacing: -.035em; font-weight: 650; }\np, [data-testid="stCaptionContainer"] { color: var(--pw-sub); }\n[data-testid="stWidgetLabel"] p { color: var(--pw-ink); font-weight: 500; }\n.top-brand { display: flex; align-items: center; gap: 10px; color: var(--pw-ink); font-size: 19px; font-weight: 750; letter-spacing: .06em; padding: 5px 0; }\n.brand-symbol { display: inline-grid; place-items: center; width: 34px; height: 34px; background: #6855c7; border-radius: 10px; color: white; font-size: 21px; letter-spacing: -.1em; padding-right: 3px; }\n.top-brand small { margin-left: 12px; font-size: 9px; letter-spacing: .15em; color: var(--pw-sub); font-weight: 550; }\n.auth-hero { position: relative; overflow: hidden; padding: 44px 38px 25px; margin: 22px 0; border-radius: 24px; min-height: 560px; background: linear-gradient(140deg,#282d63 0%,#424381 45%,#1b6269 100%); box-shadow: 0 20px 50px #1b285518; animation: pw-slide .55s ease-out both; }\n.auth-hero::before { content: \'\'; position: absolute; width: 340px; height: 340px; border-radius: 50%; border: 1px solid #ffffff0d; top: -180px; right: -100px; box-shadow: 0 0 0 60px #ffffff04,0 0 0 120px #ffffff03; pointer-events: none; }\n.hero-eyebrow { color: #d2d5f1; font-size: 10px; font-weight: 650; letter-spacing: .16em; display: flex; align-items: center; gap: 10px; }\n.hero-eyebrow span { width: 7px; height: 7px; border-radius: 50%; background: #a2e9dc; }\n.auth-hero h1 { color: #fff!important; font-size: clamp(36px,4vw,55px); font-weight: 650; line-height: 1.16; margin: 36px 0 20px; letter-spacing: -.055em; }\n.auth-hero h1 em { font-style: normal; color: #b6e9e2; }\n.auth-hero .hero-copy { font-size: 15px; line-height: 1.8; color: #dae0f2; }\n.route-illustration { margin: 24px -8px 16px; }\n.route-illustration svg { display: block; width: 100%; }\n.route-line { animation: pw-route 24s linear infinite; }\n.hero-bottom { display: flex; justify-content: space-between; gap: 16px; padding-top: 20px; border-top: 1px solid #ffffff22; color: #d5e5f1; font-size: 11px; }\n[data-testid="stForm"] { background: var(--pw-card); border: 1px solid var(--pw-line); border-radius: 18px; padding: 24px; box-shadow: 0 12px 35px #1a295706; animation: pw-slide .4s ease-out both; }\n[data-testid="stTextInput"] div[data-baseweb="input"], [data-testid="stTextInput"] div[data-baseweb="base-input"], [data-testid="stSelectbox"] div[data-baseweb="select"] > div { background: var(--pw-card)!important; color: var(--pw-ink)!important; border-color: var(--pw-line)!important; border-radius: 9px; }\n[data-testid="stTextInput"] input { background: var(--pw-card)!important; color: var(--pw-ink)!important; -webkit-text-fill-color: var(--pw-ink)!important; min-height: 44px; }\n[data-testid="stTextInput"] input::placeholder { color: var(--pw-sub)!important; -webkit-text-fill-color: var(--pw-sub)!important; opacity: .8; }\n[data-testid="stTextInput"] div[data-baseweb="input"]:focus-within { border-color: var(--pw-accent)!important; box-shadow: 0 0 0 3px #8972dc1a; }\n[data-testid="stButton"] button, [data-testid="stFormSubmitButton"] button { min-height: 42px; border-radius: 9px; border: 1px solid var(--pw-line); background: var(--pw-card); color: var(--pw-ink); transition: transform .18s ease, box-shadow .18s ease; }\n[data-testid="stButton"] button p, [data-testid="stFormSubmitButton"] button p { color: inherit!important; font-weight: 550; font-size: 14px; }\n[data-testid="stButton"] button[kind^="primary"], [data-testid="stFormSubmitButton"] button[kind^="primary"] { background: #6652c7; border-color: #6652c7; color: #fff; }\n[data-testid="stButton"] button:hover:not(:disabled), [data-testid="stFormSubmitButton"] button:hover:not(:disabled) { transform: translateY(-1px); box-shadow: 0 4px 12px #54449b22; border-color: var(--pw-accent); }\nbutton:focus-visible { outline: 3px solid var(--pw-accent)!important; outline-offset: 3px; }\nbutton:disabled { opacity: .5; cursor: not-allowed; }\n[data-testid="stSidebar"] { background: var(--pw-card); border-right: 1px solid var(--pw-line); }\n[data-testid="stSidebar"] [data-testid="stSidebarContent"] { padding-top: 1.2rem; }\n[data-testid="stRadio"] label p { color: var(--pw-ink)!important; }\n[data-testid="stMetric"] { background: var(--pw-card); border: 1px solid var(--pw-line); border-radius: 14px; padding: 18px; }\n[data-testid="stMetricValue"] { color: var(--pw-ink); font-size: clamp(21px,2.4vw,30px); }\n[data-testid="stMetricLabel"] p { color: var(--pw-sub); }\n.workspace-heading { display: flex; align-items: center; justify-content: space-between; gap: 15px; margin: 18px 0 14px; }\n.workspace-heading h2 { font-size: 30px; margin: 4px 0 0; }\n.eyebrow { font-size: 10px; letter-spacing: .15em; font-weight: 650; color: var(--pw-sub); }\n.role-badge { border: 1px solid var(--pw-line); border-radius: 20px; padding: 7px 12px; background: var(--pw-card); color: var(--pw-accent); font-size: 12px; font-weight: 650; }\n.welcome-panel { padding: 28px 30px; border-radius: 18px; background: linear-gradient(110deg,var(--pw-soft),var(--pw-card)); border: 1px solid var(--pw-line); margin-bottom: 18px; animation: pw-slide .45s ease-out both; }\n.welcome-panel h2 { font-size: 31px; margin: 9px 0; }\n.welcome-panel p { margin: 0; font-size: 14px; }\n.module-grid { display: grid; grid-template-columns: repeat(2,minmax(0,1fr)); gap: 16px; margin-bottom: 20px; }\n.module-card { background: var(--pw-card); border: 1px solid var(--pw-line); border-radius: 14px; padding: 22px; }\n.module-label { font-size: 9px; letter-spacing: .12em; color: var(--pw-sub); }\n.module-card h4 { font-size: 18px; margin: 10px 0 7px; }\n.module-card p { font-size: 13px; line-height: 1.65; margin: 0; }\n.footer-note { text-align: center; color: var(--pw-sub); font-size: 11px; margin: 28px 0 10px; }\n@keyframes pw-slide { from { opacity: 0; transform: translateY(12px); } to { opacity: 1; transform: translateY(0); } }\n@keyframes pw-route { to { stroke-dashoffset: -120; } }\n@media (max-width: 1000px) { .top-brand small { display: none; } .block-container { padding: 1.3rem 1.4rem; } .auth-hero { padding: 28px 24px; min-height: 500px; } }\n@media (max-width: 640px) { .block-container { padding: 1rem; } .auth-hero { min-height: 0; margin: 8px 0; padding: 24px; } .auth-hero h1 { font-size: 35px; margin: 20px 0 12px; } .route-illustration, .hero-bottom { display: none; } .hero-copy { margin-bottom: 0; } .hero-eyebrow { font-size: 8px; } .module-grid { grid-template-columns: 1fr; } .workspace-heading h2 { font-size: 25px; } .role-badge { font-size: 10px; } }\n@media (prefers-reduced-motion: reduce) { *,*::before,*::after { animation: none!important; transition: none!important; scroll-behavior: auto!important; } }\n\n[data-testid="stCaptionContainer"] p { color: var(--pw-sub)!important; opacity: 1; }\n[data-testid="stTextInput"] button { background: var(--pw-soft); color: var(--pw-ink); }\n[data-testid="stTextInput"] svg { fill: currentColor; }\n@media (max-width: 640px) {\n  .st-key-topbar [data-testid="stHorizontalBlock"] { flex-direction: row!important; gap: 8px; }\n  .st-key-topbar [data-testid="stColumn"] { min-width: 0!important; }\n  .st-key-topbar [data-testid="stColumn"]:first-child { flex: 1.5 1 0!important; }\n  .st-key-topbar [data-testid="stColumn"]:not(:first-child) { flex: 1 1 0!important; }\n  .st-key-topbar button p { font-size: 11px!important; }\n  .top-brand { font-size: 14px; gap: 6px; letter-spacing: .02em; }\n  .brand-symbol { width: 26px; height: 28px; font-size: 17px; flex-shrink: 0; }\n  .auth-hero { padding: 20px; }\n  .auth-hero h1 { font-size: 29px; margin: 0; }\n  .hero-eyebrow, .auth-hero .hero-copy { display: none; }\n}\n/* Keep deployment controls out of the end-user interface. */\n[data-testid="stToolbar"] { display: none; }\n/* FreightQuote AI: cinematic photography with restrained motion. */\n.block-container { max-width: 1440px; }\n.top-brand { letter-spacing: -.035em; font-size: 22px; }\n.brand-symbol { font-size: 14px; letter-spacing: -.06em; background: #167b83; }\n.top-brand small { letter-spacing: .12em; }\n.freight-scene { isolation: isolate; min-height: 670px; padding: 40px; background: #102f3b; box-shadow: 0 28px 65px #09242e25; animation: freight-enter .85s cubic-bezier(.2,.7,.2,1) both; }\n.freight-scene::before { display: none; }\n.freight-photo, .freight-shade { position: absolute; inset: 0; z-index: -2; }\n.freight-photo { background: url(\'/app/static/freight_port.webp\') center 58% / cover; animation: freight-drift 28s ease-in-out infinite alternate; }\n.freight-shade { z-index: -1; background: linear-gradient(180deg,rgba(4,19,32,.78),rgba(4,19,32,.12) 48%,rgba(4,19,32,.04) 70%,rgba(4,19,32,.85)); }\n.freight-copy h1 { font-size: clamp(36px,3.7vw,54px); line-height: 1.12; margin-top: 26px; }\n.freight-copy h1 em { color: #c6eee4; }\n.freight-copy .hero-copy { color: #e2edf2; font-size: 14px; }\n.freight-caption { position: absolute; bottom: 30px; left: 40px; right: 40px; display: flex; align-items: center; gap: 11px; color: #edf4f5; font-size: 9px; letter-spacing: .13em; }\n.caption-rule { height: 1px; flex: 1; background: #ffffff55; position: relative; overflow: hidden; }\n.caption-rule::after { content: \'\'; position: absolute; width: 35%; height: 1px; background: #b0f7df; animation: freight-route 5s ease-in-out infinite; }\n.signal-dot { width: 6px; height: 6px; background: #a4e6d5; border-radius: 50%; box-shadow: 0 0 12px #a4e6d5; }\n[data-testid=\'stForm\'] { padding: 28px; box-shadow: 0 18px 48px #193b5010; animation: freight-form .65s cubic-bezier(.2,.7,.2,1) both; }\n[data-testid=\'stButton\'] button[kind^=\'primary\'],[data-testid=\'stFormSubmitButton\'] button[kind^=\'primary\'] { background: #167b83; border-color: #167b83; min-height: 47px; }\n@keyframes freight-enter { from { opacity: 0; transform: translateX(-28px); } to { opacity: 1; transform: translateX(0); } }\n@keyframes freight-form { from { opacity: 0; transform: translateX(24px); } to { opacity: 1; transform: translateX(0); } }\n@keyframes freight-drift { from { transform: scale(1.02) translateY(0); } to { transform: scale(1.08) translateY(-8px); } }\n@keyframes freight-route { from { transform: translateX(-110%); } to { transform: translateX(400%); } }\n@media(max-width:640px) {\n .top-brand { font-size: 14px; } .top-brand .brand-symbol { width: 25px; font-size: 11px; }\n .freight-scene { min-height: 245px; padding: 24px; }\n .freight-copy h1 { font-size: 30px; margin: 0; }\n .freight-caption { left: 24px; right: 24px; bottom: 20px; font-size: 7px; }\n .freight-photo { background-position: center 62%; }\n [data-testid=\'stForm\'] { padding: 22px; }\n}\n@media(prefers-reduced-motion:reduce) { .freight-scene,.freight-photo,.caption-rule::after,[data-testid=\'stForm\'] { animation:none!important; } }\n\n/* Blended split composition; presentation only. */\n[data-testid="stApp"] { background: radial-gradient(ellipse at 12% 12%,#98d2ce28,transparent 50%),radial-gradient(ellipse at 90% 80%,#8ab7cf26,transparent 50%),var(--pw-bg); }\n.st-key-topbar { padding: 8px 4px 22px; }\n.st-key-auth_shell { position:relative; border-radius:32px; padding:12px; overflow:hidden; background:linear-gradient(115deg,#103846 0%,#26535e 48%,var(--pw-soft) 78%); border:1px solid #ffffff35; box-shadow:0 28px 75px #092b3b20; }\n.st-key-auth_shell > div { position:relative; }\n.st-key-auth_shell .freight-scene { min-height:680px; margin:0; border-radius:23px 8px 8px 23px; box-shadow:none; background:transparent; padding:44px 36px; }\n.st-key-auth_shell .freight-photo { background-position:center 60%; }\n.st-key-auth_shell .freight-shade { background:linear-gradient(180deg,#061d2de0,transparent 62%,#0b2c3cdb),linear-gradient(90deg,transparent 65%,#26535e75); }\n.st-key-auth_shell .freight-copy h1 { font-size:clamp(36px,3.5vw,53px); font-weight:550; letter-spacing:-.055em; margin-top:30px; }\n.st-key-auth_shell .hero-eyebrow { font-size:9px; letter-spacing:.2em; color:#c5e6e7; }\n.st-key-auth_shell .hero-copy { max-width:280px; line-height:1.7; }\n.st-key-auth_glass { margin:28px 22px 28px 10px; padding:36px 30px 26px; width:calc(100% - 32px)!important; border-radius:24px; background:color-mix(in srgb,var(--pw-card) 86%,transparent); border:1px solid color-mix(in srgb,var(--pw-card) 80%,transparent); backdrop-filter:blur(22px); -webkit-backdrop-filter:blur(22px); box-shadow:0 20px 65px #08293620,inset 0 1px 0 #ffffff50; animation:freight-form .8s cubic-bezier(.2,.7,.2,1) both; }\n.st-key-auth_glass h3 { font-size:32px; font-weight:600; letter-spacing:-.04em; }\n.st-key-auth_glass [data-testid="stForm"] { padding:10px 0 4px; border:0; border-radius:0; background:transparent; box-shadow:none; animation:none; }\n.st-key-auth_glass [data-testid="stTextInput"] div[data-baseweb="input"] { border-radius:11px; }\n.st-key-auth_glass [data-testid="stTextInput"] input { min-height:48px; }\n.st-key-auth_glass [data-testid="stFormSubmitButton"] button { margin-top:10px; background:linear-gradient(100deg,#11666e,#208b91); min-height:50px; border:0; box-shadow:0 6px 18px #167b8324; }\n.st-key-auth_glass [data-testid="stExpander"] { margin-top:2px; }\n.st-key-auth_glass [data-testid="stExpander"] details { border-color:var(--pw-line); background:transparent; border-radius:10px; }\n@media(max-width:1000px) and (min-width:641px) { .st-key-auth_glass {padding:26px 20px;margin:20px 8px;width:calc(100% - 16px)!important;} .st-key-auth_shell .freight-scene {padding:35px 25px;} }\n@media(max-width:640px) { .st-key-topbar {padding-bottom:12px;} .st-key-auth_shell {padding:8px;border-radius:24px;} .st-key-auth_shell .freight-scene {min-height:265px;border-radius:18px;padding:26px;} .st-key-auth_shell .freight-copy h1 {font-size:31px;margin-top:0;} .st-key-auth_glass {margin:4px 0 0;padding:26px 20px;width:100%!important;border-radius:18px;} .st-key-auth_glass h3 {font-size:28px;} .cargo-route {left:26px;right:26px;bottom:54px;height:35px;} .freight-caption {left:26px;right:26px;} .st-key-topbar button p {font-size:9px!important;} }\n@media(prefers-reduced-motion:reduce) { .cargo-vessel {animation:none!important;left:45%;opacity:.9;} .cargo-vessel svg {animation:none!important;} }\n\n/* Final editorial presentation. */\n.st-key-auth_shell {padding:0;background:var(--pw-card);border:1px solid var(--pw-line);}\n.st-key-auth_shell .freight-scene {min-height:740px;border-radius:0;margin:0;}\n.freight-photo {animation:none;transform:none;}\n.st-key-auth_shell .freight-shade {background:linear-gradient(180deg,rgba(4,20,29,.04) 20%,rgba(4,20,29,.15) 45%,rgba(4,20,29,.92));}\n.freight-copy {position:absolute;bottom:48px;left:36px;right:30px;z-index:2;}\n.st-key-auth_shell .freight-copy h1 {font-family:Georgia,serif;font-size:clamp(38px,3.6vw,54px);line-height:1.14;font-weight:500;letter-spacing:-.045em;margin:18px 0;}\n.freight-copy h1 em {font-style:italic;}\n.st-key-auth_shell .hero-copy {max-width:360px;}\n.st-key-auth_glass {background:transparent;border:0;box-shadow:none;backdrop-filter:none;padding:40px;}\n.st-key-auth_glass h3 {font-family:Georgia,serif;font-size:38px;font-weight:500;}\n.st-key-auth_glass [data-testid="stCaptionContainer"] p {color:var(--pw-sub)!important;}\n.freight-scene::after {content:\'\';position:absolute;inset:-50%;pointer-events:none;z-index:1;background:linear-gradient(110deg,transparent 38%,rgba(210,245,240,.025) 44%,rgba(255,248,225,.13) 50%,rgba(210,245,240,.025) 56%,transparent 62%);transform:translateX(-50%);animation:photo-sheen 18s ease-in-out infinite;}\n@keyframes photo-sheen {0%,15% {transform:translateX(-50%);opacity:0;}30% {opacity:.8;}65% {opacity:.8;}80%,100% {transform:translateX(50%);opacity:0;}}\n@media(max-width:640px) {.st-key-auth_shell .freight-scene {min-height:330px;}.freight-copy {bottom:24px;left:24px;right:24px;}.st-key-auth_shell .freight-copy h1 {font-size:33px;}.st-key-auth_glass {padding:26px 22px;}.st-key-auth_glass h3 {font-size:32px;}}\n@media(prefers-reduced-motion:reduce) {.freight-scene::after {animation:none!important;opacity:.3;transform:none;}}\n\n/* Theme the complete input shell, including the reveal-button wrapper. */\n[data-testid="stTextInput"] [data-baseweb="input"] {background:var(--pw-card)!important;border:1px solid var(--pw-line)!important;overflow:hidden;}\n[data-testid="stTextInput"] [data-baseweb="input"] > div,\n[data-testid="stTextInput"] [data-baseweb="base-input"],\n[data-testid="stTextInput"] [data-baseweb="input"] button {background:var(--pw-card)!important;color:var(--pw-ink)!important;}\n[data-testid="stTextInput"] [data-baseweb="input"] button {align-self:stretch;border:0!important;border-radius:0!important;min-width:44px;box-shadow:none!important;}\n[data-testid="stTextInput"] [data-baseweb="input"] button:hover {background:var(--pw-soft)!important;}\n[data-testid="stTextInput"] [data-baseweb="input"]:focus-within {border-color:var(--pw-accent)!important;box-shadow:0 0 0 3px #167b8320!important;}\n[data-testid="stSelectbox"] [data-baseweb="select"] > div,\n[data-testid="stSelectbox"] [role="combobox"] {background:var(--pw-card)!important;color:var(--pw-ink)!important;border-color:var(--pw-line)!important;}\n[data-testid="stSelectbox"] svg {color:var(--pw-ink)!important;fill:currentColor!important;}\n[data-baseweb="popover"] [role="listbox"], [data-baseweb="popover"] ul {background:var(--pw-card)!important;color:var(--pw-ink)!important;}\n[data-baseweb="popover"] [role="option"] {background:var(--pw-card)!important;color:var(--pw-ink)!important;}\n[data-baseweb="popover"] [role="option"]:hover,[data-baseweb="popover"] [aria-selected="true"] {background:var(--pw-soft)!important;}\n.st-key-auth_glass [data-testid="stCaptionContainer"] {opacity:1!important;}\n.st-key-auth_glass [data-testid="stCaptionContainer"] p {color:var(--pw-sub)!important;}\n\n/* Cover nested Streamlit control wrappers, independent of BaseWeb markup. */\n.st-key-auth_glass [data-testid="stTextInput"] div,\n.st-key-auth_glass [data-testid="stTextInput"] button,\n.st-key-auth_glass [data-testid="stSelectbox"] div,\n.st-key-auth_glass [data-testid="stSelectbox"] button {\n background-color:var(--pw-card)!important;\n background-image:none!important;\n color:var(--pw-ink)!important;\n border-color:var(--pw-line)!important;\n}\n.st-key-auth_glass [data-testid="stTextInput"] button {\n height:100%!important;min-height:48px!important;\n align-self:stretch!important;box-shadow:none!important;\n}\n.st-key-auth_glass [data-testid="stSelectbox"] span,\n.st-key-auth_glass [data-testid="stSelectbox"] input {\n color:var(--pw-ink)!important;-webkit-text-fill-color:var(--pw-ink)!important;\n}\n.st-key-auth_glass [data-testid="stTextInput"] svg,\n.st-key-auth_glass [data-testid="stSelectbox"] svg {\n color:var(--pw-sub)!important;fill:currentColor!important;\n}\n.st-key-auth_glass [data-testid="stTextInput"] div:focus-within {\n border-color:var(--pw-accent)!important;\n}\n\n/* Brief editorial text entrance, adapted from the supplied reference. */\n.st-key-auth_glass {animation:none;}\n.fq-intro .fq-kicker {font-size:11px;font-weight:650;letter-spacing:.18em;color:var(--pw-accent)!important;margin:0 0 10px;animation:fq-text-rise .7s cubic-bezier(.2,.7,.2,1) both;}\n.st-key-auth_glass .fq-intro h3 {font-family:Georgia,"Times New Roman",serif;font-size:clamp(30px,3vw,42px);font-weight:600;line-height:1.15;margin:0 0 4px;letter-spacing:-.04em;animation:fq-text-rise .7s cubic-bezier(.2,.7,.2,1) .15s both;}\n.st-key-auth_glass > div > [data-testid="stElementContainer"]:has([data-testid="stCaptionContainer"]) {animation:fq-text-rise .7s cubic-bezier(.2,.7,.2,1) .24s both;}\n.freight-copy .hero-eyebrow {animation:fq-text-rise .7s cubic-bezier(.2,.7,.2,1) .06s both;}\n.freight-copy h1 {animation:fq-text-rise .7s cubic-bezier(.2,.7,.2,1) .15s both;}\n.freight-copy .hero-copy {animation:fq-text-rise .7s cubic-bezier(.2,.7,.2,1) .24s both;}\n@keyframes fq-text-rise {from {opacity:0;transform:translateY(12px);filter:blur(4px);}to {opacity:1;transform:translateY(0);filter:blur(0);}}\n@media(prefers-reduced-motion:reduce) {.fq-intro *,.freight-copy *, .st-key-auth_glass [data-testid="stElementContainer"] {animation:none!important;filter:none!important;transform:none!important;opacity:1!important;}}\n', '.streamlit/config.toml': '[theme]\nbase="light"\nprimaryColor="#16A3B1"\nbackgroundColor="#F2F8FB"\ntextColor="#102B37"\n\n[server]\nheadless=true\nenableStaticServing=true\nenableXsrfProtection=true\nenableCORS=true\nmaxUploadSize=10\n\n[browser]\ngatherUsageStats=false\n', 'tests/test_app.py': 'import os\nimport re\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nfrom streamlit.testing.v1 import AppTest\nfrom auth_core import Store, QUESTIONS\n\nSECRET = "ui-test-key-" * 5\nPASSWORD = "StrongPwd1!"\nADMIN_PASSWORD = "AdminForTests123!"\nROOT = Path(__file__).resolve().parents[1]\n\n\nclass AppTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.env = patch.dict(\n            os.environ,\n            {\n                "JWT_SECRET": SECRET,\n                "ADMIN_PASSWORD": ADMIN_PASSWORD,\n                "ADMIN_USERNAME": "admin",\n                "DATABASE_PATH": str(Path(self.tmp.name) / "auth.db"),\n                "REQUIRE_EMAIL_VERIFICATION": "0",\n                "EMAIL_ADDRESS": "sender@example.com",\n                "EMAIL_PASSWORD": "test-only",\n            },\n        )\n        self.env.start()\n        self.smtp = patch("smtplib.SMTP_SSL")\n        self.smtp_mock = self.smtp.start()\n        self.smtp_mock.return_value.__enter__.return_value.send_message.return_value = (\n            {}\n        )\n        self.at = AppTest.from_file(str(ROOT / "app.py"), default_timeout=25).run()\n        self.store = Store(os.environ["DATABASE_PATH"])\n        self.assert_clean()\n\n    def tearDown(self):\n        self.smtp.stop()\n        self.env.stop()\n        self.tmp.cleanup()\n\n    def assert_clean(self):\n        self.assertEqual(len(self.at.exception), 0, str(self.at.exception))\n\n    def click(self, label):\n        next(x for x in self.at.button if x.label == label).click().run()\n        self.assert_clean()\n\n    def field(self, label, value):\n        next(x for x in self.at.text_input if x.label == label).set_value(value)\n\n    def seed(self):\n        self.assertTrue(\n            self.store.signup(\n                "alex", "alex@example.com", PASSWORD, QUESTIONS[0], "My School"\n            )[0]\n        )\n\n    def test_lock_closes_recovery_with_one_notice(self):\n        self.seed()\n        self.click("Forgot password?")\n        self.field("Registered email", "alex@example.com")\n        self.click("Send code")\n        email = (\n            self.smtp_mock.return_value.__enter__.return_value.send_message.call_args[\n                0\n            ][0]\n        )\n        code = re.search(r"\\b[0-9]{6}\\b", email.get_content()).group()\n        wrong = "000001" if code != "000001" else "000002"\n        for _ in range(4):\n            self.store.verify_otp("alex@example.com", wrong, SECRET)\n        self.field("Six-digit OTP", wrong)\n        self.click("Verify code")\n        self.assertTrue(self.store.is_locked("alex"))\n        self.assertEqual(sum("Account locked" in e.value for e in self.at.error), 1)\n        self.assertFalse(\n            any(\n                x.label in ("Verify code", "Send code", "Resend code")\n                for x in self.at.button\n            )\n        )\n        self.assertFalse(any(x.label == "Six-digit OTP" for x in self.at.text_input))\n        self.click("Back to sign in")\n\n    def test_motion_controls_and_theme(self):\n        self.assertTrue(self.at.session_state["motion_enabled"])\n        self.click("Pause motion")\n        self.assertFalse(self.at.session_state["motion_enabled"])\n        self.click("☾ Dark")\n        self.assertEqual(self.at.session_state["theme"], "Dark")\n        self.assertFalse(self.at.session_state["motion_enabled"])\n        self.click("Resume motion")\n        self.assertTrue(self.at.session_state["motion_enabled"])\n        self.click("☀ Light")\n        self.assertEqual(self.at.session_state["theme"], "Light")\n\n    def test_signup_blank_answer_and_user_login(self):\n        self.click("Create account")\n        for label, value in [\n            ("Username", "alex"),\n            ("Email address", "alex@example.com"),\n            ("Password", PASSWORD),\n            ("Confirm password", PASSWORD),\n            ("Security answer", "    "),\n        ]:\n            self.field(label, value)\n        self.at.selectbox[0].select(QUESTIONS[0])\n        self.click("Create account →")\n        self.assertIsNone(self.store.user("alex"))\n        self.assertTrue(len(self.at.error))\n        self.field("Security answer", "My School")\n        self.click("Create account →")\n        self.field("Username or email", "alex")\n        self.field("Password", PASSWORD)\n        self.click("Sign in →")\n        self.assertTrue(any(x.label == "Log out" for x in self.at.button))\n        self.click("Log out")\n        self.assertTrue(any(x.label == "Sign in →" for x in self.at.button))\n\n    def test_otp_reset_and_admin_unlock(self):\n        self.seed()\n        self.click("Forgot password?")\n        self.field("Registered email", "alex@example.com")\n        self.click("Send code")\n        email = (\n            self.smtp_mock.return_value.__enter__.return_value.send_message.call_args[\n                0\n            ][0]\n        )\n        code = re.search(r"\\b[0-9]{6}\\b", email.get_content()).group()\n        self.assertTrue(\n            next(x for x in self.at.button if x.label == "Resend code").disabled\n        )\n        self.field("Six-digit OTP", code)\n        self.click("Verify code")\n        self.field("New password", "NewStrongPwd2!")\n        self.field("Confirm new password", "NewStrongPwd2!")\n        self.click("Reset password")\n        self.assertIsNotNone(self.store.authenticate("alex", "NewStrongPwd2!")[0])\n        # Lock the same account, then unlock it using the actual admin interface.\n        code, _ = self.store.begin_otp("alex@example.com", SECRET)\n        wrong = "000001" if code != "000001" else "000002"\n        for _ in range(5):\n            self.store.verify_otp("alex@example.com", wrong, SECRET)\n        self.field("Username or email", "admin")\n        self.field("Password", ADMIN_PASSWORD)\n        self.click("Sign in →")\n        self.at.radio[0].set_value("User management").run()\n        self.assert_clean()\n        self.assertTrue(self.store.is_locked("alex"))\n        self.assertTrue(len(self.at.dataframe))\n        self.click("Unlock account")\n        self.assertFalse(self.store.is_locked("alex"))\n\n    def test_single_login_role_navigation_and_revocation(self):\n        from auth_core import issue_token\n\n        self.seed()\n        self.assertFalse(any("Admin sign" in x.label for x in self.at.button))\n        root = issue_token("admin", SECRET, role="admin", kind="configured_admin")\n        for role in ("operations_manager", "freight_broker", "auditor", "admin"):\n            self.assertTrue(self.store.assign_role("alex", role, root, SECRET)[0])\n            self.field("Username or email", "alex@example.com")\n            self.field("Password", PASSWORD)\n            self.click("Sign in →")\n            choices = self.at.radio[0].options\n            self.assertEqual("User management" in choices, role == "admin")\n            self.assertEqual("Audit trail" in choices, role in ("admin", "auditor"))\n            self.at.radio[0].set_value("My profile").run()\n            self.assert_clean()\n            self.at.radio[0].set_value("Security").run()\n            self.assert_clean()\n            self.assertTrue(\n                any(x.label == "Current password" for x in self.at.text_input)\n            )\n            self.click("Log out")\n        self.field("Username or email", "alex")\n        self.field("Password", PASSWORD)\n        self.click("Sign in →")\n        self.assertTrue(self.store.assign_role("alex", "customer", root, SECRET)[0])\n        self.at.run()\n        self.assert_clean()\n        self.assertTrue(any(x.label == "Sign in →" for x in self.at.button))\n        self.assertFalse(any(x.label == "Log out" for x in self.at.button))\n\n    def test_security_question_rejects_blank_then_resets(self):\n        self.seed()\n        self.click("Forgot password?")\n        self.at.radio[0].set_value("Security question").run()\n        self.assert_clean()\n        self.field("Registered username", "alex")\n        self.click("Find security question")\n        self.field("Your answer", "    ")\n        self.click("Verify answer")\n        self.assertTrue(\n            any("Enter a security answer" in e.value for e in self.at.error)\n        )\n        self.field("Your answer", "  MY SCHOOL  ")\n        self.click("Verify answer")\n        self.field("New password", "NewStrongPwd2!")\n        self.field("Confirm new password", "NewStrongPwd2!")\n        self.click("Reset password")\n        self.assertIsNotNone(self.store.authenticate("alex", "NewStrongPwd2!")[0])\n\n    def test_signup_activation_email_flow(self):\n        os.environ["REQUIRE_EMAIL_VERIFICATION"] = "1"\n        self.click("Create account")\n        for label, value in [\n            ("Username", "newuser"),\n            ("Email address", "new@example.com"),\n            ("Password", PASSWORD),\n            ("Confirm password", PASSWORD),\n            ("Security answer", "School"),\n        ]:\n            self.field(label, value)\n        self.at.selectbox[0].select(QUESTIONS[0])\n        self.click("Create account →")\n        self.assertEqual(self.store.user("newuser")["email_verified"], 0)\n        mail = (\n            self.smtp_mock.return_value.__enter__.return_value.send_message.call_args[\n                0\n            ][0]\n        )\n        code = re.search(r"\\b[0-9]{6}\\b", mail.get_content()).group()\n        self.field("Six-digit code", code)\n        self.click("Verify email")\n        self.assertEqual(self.store.user("newuser")["email_verified"], 1)\n        self.assertTrue(any(x.label == "Sign in →" for x in self.at.button))\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_roles.py': '"""Authorization is enforced by the store, not just by visible navigation."""\n\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom auth_core import Store, QUESTIONS, ROLES, can, issue_token, admin_credential_tag\n\nSECRET = "test-roles-signing-key-" * 3\nPASSWORD = "TestPassword123!"\nADMIN_PASSWORD = "TestAdministrator123!"\n\n\nclass RoleTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.store = Store(Path(self.tmp.name) / "accounts.db")\n        self.store.signup(\n            "member", "member@example.com", PASSWORD, QUESTIONS[0], "School"\n        )\n        self.root = issue_token(\n            "admin",\n            SECRET,\n            role="admin",\n            kind="configured_admin",\n            credential_tag=admin_credential_tag(ADMIN_PASSWORD, SECRET),\n        )\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def token(self, username="member"):\n        row = self.store.user(username)\n        return issue_token(\n            row["email"], SECRET, role=row["role"], version=row["session_version"]\n        )\n\n    def assign(self, role, username="member", token=None):\n        return self.store.assign_role(\n            username, role, token or self.root, SECRET, "admin", ADMIN_PASSWORD\n        )\n\n    def test_one_login_and_customer_default(self):\n        member, error = self.store.sign_in("member@example.com", PASSWORD)\n        self.assertIsNone(error)\n        self.assertEqual(member["role"], "customer")\n        admin, error = self.store.sign_in(\n            " ADMIN ", ADMIN_PASSWORD, "admin", ADMIN_PASSWORD\n        )\n        self.assertIsNone(error)\n        self.assertEqual(admin["role"], "admin")\n        self.assertEqual(admin["principal_type"], "configured_admin")\n        self.assertEqual(\n            self.store.sign_in("admin", "wrong", "admin", ADMIN_PASSWORD)[1],\n            self.store.sign_in("missing", "wrong")[1],\n        )\n        self.assertIsNone(self.store.sign_in("admin", ADMIN_PASSWORD)[0])\n\n    def test_reserved_names(self):\n        self.assertFalse(\n            self.store.signup(\n                "ADMIN", "other@example.com", PASSWORD, QUESTIONS[0], "School"\n            )[0]\n        )\n        self.assertFalse(\n            self.store.signup(\n                "other",\n                "owner@example.com",\n                PASSWORD,\n                QUESTIONS[0],\n                "School",\n                reserved_identity="owner@example.com",\n            )[0]\n        )\n\n    def test_all_roles_and_session_invalidation(self):\n        previous = self.token()\n        for role in ROLES:\n            if role == "customer":\n                continue\n            self.assertTrue(self.assign(role)[0])\n            self.assertIsNone(self.store.session(previous, SECRET))\n            row, err = self.store.sign_in("member", PASSWORD)\n            self.assertIsNone(err)\n            self.assertEqual(row["role"], role)\n            previous = self.token()\n            self.assertEqual(self.store.session(previous, SECRET)["role"], role)\n        self.assertTrue(self.assign("customer")[0])\n        self.assertIsNone(self.store.session(previous, SECRET))\n\n    def test_forged_role_and_invalid_kind_rejected(self):\n        forged = issue_token("member@example.com", SECRET, role="admin")\n        self.assertIsNone(self.store.session(forged, SECRET))\n        self.assertFalse(self.assign("admin", token=self.token())[0])\n        self.assertFalse(self.assign("superadmin")[0])\n        self.assertIsNone(\n            self.store.session(issue_token("admin", SECRET, role="admin"), SECRET)\n        )\n        self.assertIsNone(\n            self.store.session(self.root, SECRET, "someone-else", ADMIN_PASSWORD)\n        )\n        self.assertIsNone(\n            self.store.session(self.root, SECRET, "admin", "ChangedPassword123!")\n        )\n\n    def test_read_permissions(self):\n        for role in ("customer", "operations_manager", "freight_broker", "auditor"):\n            if role != "customer":\n                self.assertTrue(self.assign(role)[0])\n            token = self.token()\n            with self.assertRaises(PermissionError):\n                self.store.visible_users(token, SECRET)\n            if role == "auditor":\n                self.assertTrue(self.store.audit_events(token, SECRET))\n            else:\n                with self.assertRaises(PermissionError):\n                    self.store.audit_events(token, SECRET)\n            self.assertFalse(self.store.unlock_account("member", token, SECRET)[0])\n        self.assertTrue(\n            self.store.visible_users(self.root, SECRET, "admin", ADMIN_PASSWORD)\n        )\n        self.assertFalse(can({"role": "unknown"}, "workspace"))\n\n    def test_registered_admin_and_demotion(self):\n        self.assertTrue(self.assign("admin")[0])\n        admin_token = self.token()\n        self.store.signup(\n            "another", "another@example.com", PASSWORD, QUESTIONS[0], "School"\n        )\n        self.assertTrue(self.assign("auditor", "another", admin_token)[0])\n        self.assertFalse(self.assign("customer", "member", admin_token)[0])\n        with self.store.db() as c:\n            c.execute("UPDATE users SET account_locked=1 WHERE username=\'another\'")\n        self.assertTrue(self.store.unlock_account("another", admin_token, SECRET)[0])\n        self.assertTrue(self.assign("customer")[0])\n        self.assertFalse(self.assign("admin", "another", admin_token)[0])\n        self.assertFalse(self.store.unlock_account("another", admin_token, SECRET)[0])\n\n    def test_unverified_or_locked_cannot_be_promoted(self):\n        self.store.signup(\n            "pending",\n            "pending@example.com",\n            PASSWORD,\n            QUESTIONS[0],\n            "School",\n            require_verification=True,\n        )\n        self.assertFalse(self.assign("admin", "pending")[0])\n        with self.store.db() as c:\n            c.execute("UPDATE users SET account_locked=1 WHERE username=\'member\'")\n        self.assertFalse(self.assign("admin")[0])\n        self.assertIsNone(self.store.session(self.token(), SECRET))\n\n    def test_transaction_rechecks_authorization(self):\n        self.assertTrue(self.assign("admin")[0])\n        stale_account = self.store.session(self.token(), SECRET)\n        self.assertTrue(self.assign("customer")[0])\n        with self.store.db() as c:\n            self.assertFalse(\n                self.store._still_authorized(c, stale_account, "manage_access")\n            )\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_auth_core.py': 'import concurrent.futures\nimport sqlite3\nimport tempfile\nimport time\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\nfrom auth_core import (\n    Store,\n    QUESTIONS,\n    answer_error,\n    password_errors,\n    issue_token,\n    decode_token,\n    valid_email,\n)\n\nSECRET = "test-signing-key-" * 4\nPASSWORD = "StrongPwd1!"\nNEW_PASSWORD = "NewStrongPwd2!"\n\n\nclass AuthTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.path = Path(self.tmp.name) / "auth.db"\n        self.store = Store(self.path)\n        self.assertTrue(\n            self.store.signup(\n                "alex", "alex@example.com", PASSWORD, QUESTIONS[0], "My School"\n            )[0]\n        )\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def admin_token(self):\n        return issue_token("admin", SECRET, role="admin", kind="configured_admin")\n\n    def wrong_code(self, code):\n        return "000001" if code != "000001" else "000002"\n\n    def lock(self):\n        code, _ = self.store.begin_otp("alex@example.com", SECRET)\n        for _ in range(5):\n            self.store.verify_otp("alex@example.com", self.wrong_code(code), SECRET)\n\n    def test_resend_schedule_and_budget(self):\n        now = 1000000\n        with patch("auth_core.time.time", side_effect=lambda: now):\n            for number in range(1, 6):\n                code, err = self.store.begin_otp("alex@example.com", SECRET)\n                self.assertIsNone(err)\n                self.assertRegex(code, r"^[0-9]{6}$")\n                delay = 60 if number <= 3 else 300\n                self.assertEqual(self.store.code_status("alex")["resend_in"], delay)\n                self.assertIsNone(self.store.begin_otp("alex@example.com", SECRET)[0])\n                now += delay - 1\n                self.assertIsNone(self.store.begin_otp("alex@example.com", SECRET)[0])\n                self.assertEqual(self.store.code_status("alex")["requests"], number)\n                now += 1\n            self.assertTrue(self.store.is_locked("alex"))\n            self.assertIsNone(self.store.begin_otp("alex@example.com", SECRET)[0])\n\n    def test_fifth_code_usable(self):\n        now = 1000000\n        with patch("auth_core.time.time", side_effect=lambda: now):\n            for number in range(1, 6):\n                code, _ = self.store.begin_otp("alex@example.com", SECRET)\n                if number < 5:\n                    now += 60 if number <= 3 else 300\n            self.assertFalse(self.store.is_locked("alex"))\n            grant = self.store.verify_otp("alex@example.com", code, SECRET)\n            self.assertIsInstance(grant, str)\n            self.assertTrue(\n                self.store.reset_password("alex@example.com", NEW_PASSWORD, grant)[0]\n            )\n            self.assertEqual(self.store.code_status("alex")["requests"], 0)\n\n    def test_failures_survive_resend_and_restart(self):\n        now = 1000000\n        with patch("auth_core.time.time", side_effect=lambda: now):\n            code, _ = self.store.begin_otp("alex@example.com", SECRET)\n            for _ in range(3):\n                self.store.verify_otp("alex@example.com", self.wrong_code(code), SECRET)\n            now += 60\n            code, _ = self.store.begin_otp("alex@example.com", SECRET)\n            other = Store(self.path)\n            for _ in range(2):\n                other.verify_otp("alex@example.com", self.wrong_code(code), SECRET)\n            self.assertTrue(other.is_locked("alex"))\n            self.assertFalse(other.verify_otp("alex@example.com", code, SECRET))\n            self.assertIsNone(other.authenticate("alex", PASSWORD)[0])\n            self.assertFalse(other.verify_answer("alex", "My School"))\n\n    def test_admin_unlock(self):\n        self.lock()\n        self.assertFalse(\n            self.store.unlock_account("alex", issue_token("alex", SECRET), SECRET)[0]\n        )\n        self.assertFalse(\n            self.store.unlock_account("alex", self.admin_token(), "wrong-key")[0]\n        )\n        expired = issue_token("admin", SECRET, role="admin", lifetime=-1)\n        self.assertFalse(self.store.unlock_account("alex", expired, SECRET)[0])\n        self.assertTrue(\n            self.store.unlock_account("alex", self.admin_token(), SECRET)[0]\n        )\n        self.assertFalse(self.store.is_locked("alex"))\n        self.assertEqual(self.store.code_status("alex")["failures"], 0)\n        self.assertIsNotNone(self.store.authenticate("alex", PASSWORD)[0])\n        self.assertIsNotNone(self.store.begin_otp("alex@example.com", SECRET)[0])\n        self.assertTrue(\n            any(\n                e["event"] == "account_unlocked:alex"\n                for e in self.store.recent_events()\n            )\n        )\n\n    def test_blank_invisible_and_long_answers(self):\n        for answer in ["    ", "\\t\\n", "\\u200b\\u200b", "\\u00a0" * 4, "!!!", "é" * 40]:\n            self.assertIsNotNone(answer_error(answer))\n            self.assertFalse(\n                self.store.signup(\n                    "other", "other@example.com", PASSWORD, QUESTIONS[0], answer\n                )[0]\n            )\n            self.assertFalse(self.store.verify_answer("alex", answer))\n        self.assertTrue(self.store.verify_answer("alex", "  MY SCHOOL  "))\n\n    def test_password_rules(self):\n        for password in [\n            "StrongPwd1 ",\n            "StrongPwd1",\n            "strongpwd1!",\n            "STRONGPWD1!",\n            "StrongPwd!",\n            "Aa1!" + "é" * 40,\n        ]:\n            self.assertTrue(password_errors(password), password)\n        self.assertFalse(password_errors(PASSWORD))\n\n    def test_reset_grant_required_expiring_and_single_use(self):\n        self.assertFalse(self.store.reset_password("alex@example.com", NEW_PASSWORD)[0])\n        code, _ = self.store.begin_otp("alex@example.com", SECRET)\n        grant = self.store.verify_otp("alex@example.com", code, SECRET)\n        self.assertFalse(self.store.verify_otp("alex@example.com", code, SECRET))\n        self.assertFalse(\n            self.store.reset_password("alex@example.com", NEW_PASSWORD, "bad")[0]\n        )\n        self.assertTrue(\n            self.store.reset_password("alex@example.com", NEW_PASSWORD, grant)[0]\n        )\n        self.assertFalse(\n            self.store.reset_password("alex@example.com", "AnotherPwd3!", grant)[0]\n        )\n        self.assertIsNone(self.store.authenticate("alex", PASSWORD)[0])\n        self.assertIsNotNone(self.store.authenticate("alex", NEW_PASSWORD)[0])\n        grant = self.store.verify_answer("alex", "My School")\n        with self.store.db() as c:\n            c.execute("UPDATE reset_grants SET expires_at=?", (int(time.time()) - 1,))\n        self.assertFalse(\n            self.store.reset_password("alex@example.com", PASSWORD, grant)[0]\n        )\n\n    def test_lock_revokes_sessions_and_grants(self):\n        before = self.store.user("alex")["session_version"]\n        grant = self.store.verify_answer("alex", "My School")\n        self.lock()\n        self.assertGreater(self.store.user("alex")["session_version"], before)\n        self.assertFalse(\n            self.store.reset_password("alex@example.com", NEW_PASSWORD, grant)[0]\n        )\n        self.assertFalse(self.store.change_password("alex", PASSWORD, NEW_PASSWORD)[0])\n\n    def test_replaced_and_expired_code(self):\n        now = 1000000\n        with patch("auth_core.time.time", side_effect=lambda: now):\n            first, _ = self.store.begin_otp("alex@example.com", SECRET)\n            now += 60\n            with patch(\n                "auth_core.secrets.randbelow", return_value=(int(first) + 1) % 1000000\n            ):\n                second, _ = self.store.begin_otp("alex@example.com", SECRET)\n            self.assertFalse(self.store.verify_otp("alex@example.com", first, SECRET))\n            now += 300\n            self.assertFalse(self.store.verify_otp("alex@example.com", second, SECRET))\n\n    def test_smtp_failure_retains_limits(self):\n        self.store.begin_otp("alex@example.com", SECRET)\n        self.store.cancel_otp("alex@example.com")\n        self.assertEqual(self.store.code_status("alex")["requests"], 1)\n        self.assertIsNone(self.store.begin_otp("alex@example.com", SECRET)[0])\n\n    def test_nonempty_malformed_codes_count(self):\n        self.store.begin_otp("alex@example.com", SECRET)\n        self.assertFalse(self.store.verify_otp("alex@example.com", "", SECRET))\n        self.assertEqual(self.store.code_status("alex")["failures"], 0)\n        for _ in range(5):\n            self.store.verify_otp("alex@example.com", "abcdef", SECRET)\n        self.assertTrue(self.store.is_locked("alex"))\n\n    def test_email_activation(self):\n        self.store.signup(\n            "newuser",\n            "new@example.com",\n            PASSWORD,\n            QUESTIONS[0],\n            "School",\n            require_verification=True,\n        )\n        self.assertIsNone(self.store.authenticate("newuser", PASSWORD)[0])\n        code, _ = self.store.begin_email_verification("new@example.com", SECRET)\n        self.assertEqual(self.store.code_status("newuser", "verify")["resend_in"], 60)\n        self.assertTrue(self.store.verify_email("new@example.com", code, SECRET))\n        self.assertFalse(self.store.verify_email("new@example.com", code, SECRET))\n        self.assertIsNotNone(self.store.authenticate("newuser", PASSWORD)[0])\n\n    def test_duplicate_accounts_and_generic_login_errors(self):\n        self.assertFalse(\n            self.store.signup(\n                "ALEX", "other@example.com", PASSWORD, QUESTIONS[0], "Answer"\n            )[0]\n        )\n        self.assertFalse(\n            self.store.signup(\n                "other", "alex@example.com", PASSWORD, QUESTIONS[0], "Answer"\n            )[0]\n        )\n        self.assertEqual(\n            self.store.authenticate("alex", "bad")[1],\n            self.store.authenticate("missing", "bad")[1],\n        )\n        self.assertNotIn("password_hash", self.store.list_users()[0])\n\n    def test_login_rate_limit_shared_by_aliases(self):\n        for _ in range(5):\n            self.store.authenticate("alex", "bad")\n        self.assertIsNone(self.store.authenticate("alex@example.com", PASSWORD)[0])\n\n    def test_tokens_and_email(self):\n        token = issue_token("alex", SECRET)\n        self.assertEqual(decode_token(token, SECRET)["sub"], "alex")\n        self.assertIsNone(decode_token(token, SECRET + "bad"))\n        self.assertIsNone(\n            decode_token(issue_token("alex", SECRET, lifetime=-1), SECRET)\n        )\n        self.assertTrue(valid_email("ab@cd.ef"))\n        self.assertFalse(valid_email("a@b.c"))\n\n    def test_concurrent_sends_and_single_use(self):\n        def send(_):\n            return Store(self.path).begin_otp("alex@example.com", SECRET)[0]\n\n        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as e:\n            codes = list(e.map(send, range(3)))\n        self.assertEqual(sum(c is not None for c in codes), 1)\n        code = next(c for c in codes if c)\n\n        def verify(_):\n            return Store(self.path).verify_otp("alex@example.com", code, SECRET)\n\n        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as e:\n            grants = list(e.map(verify, range(3)))\n        self.assertEqual(sum(bool(g) for g in grants), 1)\n\n    def test_migration_preserves_accounts(self):\n        old = Path(self.tmp.name) / "old.db"\n        with sqlite3.connect(old) as c:\n            c.execute(\n                """CREATE TABLE users (id INTEGER PRIMARY KEY AUTOINCREMENT,\n                username TEXT NOT NULL UNIQUE COLLATE NOCASE,email TEXT NOT NULL UNIQUE COLLATE NOCASE,\n                password_hash TEXT NOT NULL,security_question TEXT NOT NULL,answer_hash TEXT NOT NULL,\n                created_at TEXT NOT NULL,updated_at TEXT NOT NULL,login_count INTEGER DEFAULT 0,\n                last_login TEXT,previous_login TEXT,session_version INTEGER DEFAULT 0,email_verified INTEGER DEFAULT 1)"""\n            )\n            c.execute(\n                "INSERT INTO users VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)",\n                tuple(self.store.user("alex"))[:13],\n            )\n        migrated = Store(old)\n        self.assertIsNotNone(migrated.authenticate("alex", PASSWORD)[0])\n        self.assertIsNotNone(migrated.begin_otp("alex@example.com", SECRET)[0])\n\n\nif __name__ == "__main__":\n    unittest.main()\n'}
IMAGE_DATA = '...'

def restore_project_files():
    for relative_name, source in APP_FILES.items():
        destination = PROJECT_DIR / relative_name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(source, encoding="utf-8")
    image_path = PROJECT_DIR / "static/freight_port.webp"
    image_path.parent.mkdir(parents=True, exist_ok=True)
    image_path.write_bytes(base64.b64decode(IMAGE_DATA))

restore_project_files()
print(f"Setup complete: {len(APP_FILES)} application/configuration/test files and the freight image.")
print("Project folder:", PROJECT_DIR)
print("Next: run Step 4 — Check application files.")


## 4. Check application files

Uses the absolute project path, so changing the working folder does not break the checks. Full tests are optional.

In [ ]:
import py_compile
from pathlib import Path
import streamlit, bcrypt, jwt, pandas

PROJECT_DIR = Path("/content/FreightQuote_AI_Lite")
required_files = ("app.py", "auth_core.py", "motion.css", "static/freight_port.webp", ".streamlit/config.toml")
missing_files = [name for name in required_files if not (PROJECT_DIR / name).is_file()]
if missing_files and callable(globals().get("restore_project_files")):
    restore_project_files()
    missing_files = [name for name in required_files if not (PROJECT_DIR / name).is_file()]
if missing_files:
    raise RuntimeError("Run Step 3 — Create ALL application files — first. Missing: " + ", ".join(missing_files))
for filename in ("app.py", "auth_core.py"):
    py_compile.compile(str(PROJECT_DIR / filename), doraise=True)
print("Application files and dependencies are ready.")

RUN_FULL_TESTS = False #@param {type:"boolean"}
if RUN_FULL_TESTS:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=str(PROJECT_DIR), check=True)


## 5. Check Gmail sender login

In [ ]:
import smtplib

try:
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=15) as smtp:
        smtp.login(os.environ["EMAIL_ADDRESS"], os.environ["EMAIL_PASSWORD"])
    print("Gmail login accepted.")
except smtplib.SMTPAuthenticationError:
    print("Gmail rejected the login. Check the sender address and App Password.")
except (smtplib.SMTPException, OSError):
    print("Could not reach Gmail. Check the connection and try again.")


## 6. Start FreightQuote AI

Open the printed link and keep the runtime connected.

In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen
from pyngrok import ngrok

PROJECT_DIR = Path("/content/FreightQuote_AI_Lite")
if not (PROJECT_DIR / "app.py").is_file():
    if callable(globals().get("restore_project_files")):
        restore_project_files()
    else:
        raise RuntimeError("Run Step 3 — Create ALL application files — before starting the app.")
required_secrets = ("JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_PASSWORD")
if any(not os.environ.get(name) for name in required_secrets):
    raise RuntimeError("Run Step 2 — Configure Secrets — before starting the app.")

try:
    old_pids = subprocess.check_output(
        ["fuser", "-n", "tcp", "8501"], stderr=subprocess.DEVNULL, text=True
    ).split()
except (subprocess.CalledProcessError, FileNotFoundError):
    old_pids = []
for pid in old_pids:
    if pid.isdigit() and int(pid) != os.getpid():
        try:
            os.kill(int(pid), signal.SIGTERM)
        except ProcessLookupError:
            pass
time.sleep(2)
log_path = Path("/tmp/portwise_streamlit.log")
with log_path.open("w") as log:
    streamlit_process = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app.py",
         "--server.port=8501", "--server.address=0.0.0.0"],
        stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(), cwd=str(PROJECT_DIR)
    )
healthy = False
for _ in range(45):
    if streamlit_process.poll() is not None:
        break
    try:
        with urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2) as response:
            healthy = response.status == 200
        if healthy:
            break
    except OSError:
        pass
    time.sleep(1)
if not healthy:
    raise RuntimeError("Streamlit did not start. Inspect /tmp/portwise_streamlit.log privately.")
from pyngrok import conf
from pyngrok.exception import PyngrokError
import logging

# Pass the token at startup; saving it again can fail on newer ngrok agents.
conf.get_default().auth_token = os.environ["NGROK_AUTHTOKEN"].strip()

def open_public_tunnel():
    # Reuse this runtime's tunnel rather than creating a duplicate endpoint.
    for existing in ngrok.get_tunnels():
        address = str(existing.config.get("addr", "")).rstrip("/")
        if address in ("8501", "http://localhost:8501", "http://127.0.0.1:8501", "http://0.0.0.0:8501"):
            return existing.public_url
    return ngrok.connect(addr=8501, proto="http").public_url

ngrok_logger = logging.getLogger("pyngrok.process.ngrok")
previous_level = ngrok_logger.level
try:
    ngrok_logger.setLevel(logging.ERROR)
    public_url = open_public_tunnel()
    print("Open FreightQuote AI:", public_url)
except PyngrokError as exc:
    if "ERR_NGROK_334" in str(exc):
        print("FreightQuote AI is running. Your ngrok address is already in use by another tunnel.")
        print("Stop the old endpoint at https://dashboard.ngrok.com/endpoints, or stop ngrok in the older Colab runtime. Then rerun Step 6.")
    else:
        print("FreightQuote AI is running, but ngrok could not connect. Check your ngrok account and token, then rerun Step 6.")
    print("Opening a Colab preview below so you can continue in this notebook.")
    try:
        from google.colab import output
        output.serve_kernel_port_as_iframe(8501, height=900)
    except Exception:
        print("Colab preview is unavailable in this environment. Locally, open http://localhost:8501.")
finally:
    ngrok_logger.setLevel(previous_level)


## Notes

- Account locking closes recovery and displays one notice. Only an Admin can unlock the account.
- Colab storage is temporary; preserve the database privately if accounts need to survive runtime deletion.
- Clear notebook outputs before sharing. Never share your Secret values.
- Freight business services are still planned; this notebook provides authentication and account dashboards.